# 00 · Environment, reproducibility and library self-check

**What this notebook does.** Records a full reproduction manifest (Python, torch,
CUDA, transformers, GPU model, revisions), then runs the library's *CPU-only* unit
checks so you find a broken install or a broken formula in 60 seconds instead of
40 minutes into a quantization run.

**Why this comes first.** Every later notebook writes numbers into a results table.
If the environment is not recorded, those numbers are unreproducible and the whole
study is unpublishable — that is Milestone 0 in the pivot document.

**Kaggle settings.**

| Setting | Value | Why |
|---|---|---|
| Accelerator | GPU T4 x1 (or P100) | 16 GB is enough for a 1.5B FP16 model + FP32 Hessians |
| Internet | **On** | model + dataset download |
| Persistence | Save Version -> *Save & Run All* | so later notebooks can attach this output |

**Runtime.** Under 2 minutes. No GPU work.

**Produces.** `env/env_report.json`, `logs/run_log.jsonl`

In [ ]:
# =====================================================================================
# CELL 0 -- LIBRARY AND CONFIGS (generated; identical in every notebook)
#
# Source of truth: D:\BTP code\src\ebswgptaq  (edit there, then rebuild the
# notebooks with scripts/build_notebooks.py). Do not edit this cell by hand.
# =====================================================================================
import os, sys, json

_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
PKG_DIR = os.path.join(_ROOT, "ebswgptaq")
CONFIG_DIR = os.path.join(_ROOT, "configs")
os.makedirs(PKG_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)

MODULES = {}
MODULES['__init__.py'] = r'''"""
EB-SW-GPTAQ — research library for the BTP proxy-gate study.

Modules
-------
config              : dataclass configs + YAML loading
logging_utils       : artifact paths, JSON/JSONL IO, environment manifest, seeding
model_loader        : FP16 loader with FP32 lm_head guard, ModelWrapper
batching            : CalibBatch construction, causal/valid-position bookkeeping
calibration_data    : training-split prompt sources, mixed-corpus assembly, leak audit
trace_generation    : self-generated CoT traces (deterministic primary decoding)
quant_core          : group quantizer + weighted GPTQ/GPTAQ solver (the method)
probe_quantization  : RTN / GPTQ probe checkpoints used for the JSD signal
jsd                 : FP32 log-softmax, exact + top-k JSD, KL, strata
score_weights       : score -> weights (temperature, ESS bisection, controls)
activation_capture  : hooks that return per-module calibration inputs
geometry            : covariance spectra, effective rank, principal angles, overlap
evaluation          : held-out teacher-forced KL/JSD, perplexity, generation scoring
statistics          : paired bootstrap CI, McNemar, rank correlation
quantizer_adapter   : integer-code export / packing stub for real kernels later
"""

__version__ = "0.1.0"

__all__ = ["__version__"]
'''
MODULES['activation_capture.py'] = r'''"""
Capture per-module input activations for geometry analysis (Milestone 4).

Returns a list with one [d_in, T] tensor per calibration sequence rather than one
giant matrix, because a 1.5B model with 32k calibration tokens produces ~1.2 GB per
large module and there is no reason to hold it all at once.

The activation returned by a forward hook is the module's *input*, i.e. exactly the
matrix X that the solver accumulates into the Hessian, so geometry diagnostics and
the solver see the same columns in the same order.
"""
from __future__ import annotations

from typing import Dict, List, Optional, Sequence

import torch

from .batching import CalibBatch


def capture_layer_inputs(
    wrapped,
    batches: Sequence[CalibBatch],
    layer_idx: int,
    module_filter: Optional[Sequence[str]] = None,
    store_dtype: torch.dtype = torch.float16,
) -> Dict[str, List[torch.Tensor]]:
    """
    For every linear module of one decoder layer, capture [d_in, T] inputs per batch.

    Returns {module_name: [tensor_batch0, tensor_batch1, ...]} on CPU.
    """
    mods = wrapped.layer_linears(layer_idx)
    if module_filter:
        mods = {n: m for n, m in mods.items() if any(f in n for f in module_filter)}

    out: Dict[str, List[torch.Tensor]] = {name: [] for name in mods}
    handles = []

    def make(name):
        def hook(_mod, inp, _out):
            x = inp[0]
            if x.dim() == 3:
                x = x[0]
            out[name].append(x.detach().to(store_dtype).T.contiguous().cpu())
        return hook

    for name, mod in mods.items():
        handles.append(mod.register_forward_hook(make(name)))
    try:
        with torch.no_grad():
            for batch in batches:
                wrapped.forward_hidden(batch.input_ids, batch.attention_mask)
    finally:
        for h in handles:
            h.remove()
    return out


def concat_columns(tensors: Sequence[torch.Tensor]) -> torch.Tensor:
    """[d, T1] + [d, T2] + ... -> [d, sum(T)]."""
    if not tensors:
        raise ValueError("no tensors to concatenate")
    return torch.cat(list(tensors), dim=1)


def select_positions(X: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Column subset selected by a boolean mask over columns."""
    m = mask.detach().cpu().flatten().bool()
    if m.numel() != X.shape[1]:
        raise ValueError(f"mask length {m.numel()} != {X.shape[1]} columns")
    return X[:, m]


def target_masks_per_batch(batches) -> List[torch.Tensor]:
    """
    One boolean mask per batch, True at positions that have a valid next-token target.

    Captured activations have one column per *token*, while sensitivity scores exist
    only for positions with a target, so any analysis that joins the two (geometry,
    weighting) must select columns through these masks. Getting this wrong shifts
    every score by one column and is invisible in the output.
    """
    from .batching import target_positions

    masks = []
    for b in batches:
        m = torch.zeros(b.length, dtype=torch.bool)
        pos = target_positions(b.valid_mask)
        if pos.numel():
            m[pos] = True
        masks.append(m)
    return masks


def concat_target_columns(tensors: Sequence[torch.Tensor], masks: Sequence[torch.Tensor],
                          dtype: torch.dtype = torch.float32) -> torch.Tensor:
    """Concatenate per-batch [d, T] tensors keeping only scored columns -> [d, P]."""
    if len(tensors) != len(masks):
        raise ValueError(f"{len(tensors)} tensors vs {len(masks)} masks")
    parts = [t.to(dtype)[:, m] for t, m in zip(tensors, masks)]
    return torch.cat(parts, dim=1)


def column_alignment_check(wrapped, batch: CalibBatch, layer_idx: int) -> Dict[str, object]:
    """
    Empirical alignment probe: column count must equal the number of valid tokens,
    and column t must be the activation that produced logit row t.

    The cheap invariant verified here is that the captured matrix has exactly
    `batch.n_valid` columns for a single-sequence batch, which catches double-counting
    (e.g. a hook firing twice or a [B,T,d] reshape that transposes batch and time).
    """
    caps = capture_layer_inputs(wrapped, [batch], layer_idx)
    first = next(iter(caps.values()))
    d = first[0].shape[0]
    return {
        "layer": layer_idx,
        "modules_captured": len(caps),
        "d_in": int(d),
        "columns": int(first[0].shape[1]),
        "valid_tokens": int(batch.n_valid),
        "aligned": bool(first[0].shape[1] == batch.n_valid),
    }
'''
MODULES['batching.py'] = r'''"""
Calibration batches with an explicit, testable position contract.

The single most common silent bug in this literature is a one-position misalignment
between logits, input ids and activation columns (v2 §9.2). This module fixes one
canonical definition and every other module consumes it:

    A CalibBatch has exactly one sequence (B = 1) of length T.
    `valid_mask[t]`  is True iff token t is a real (non-padding) token.
    Activation column t of every linear module corresponds to token t.
    Logit row t predicts token t+1 (causal LM convention).

Therefore:
    calibration columns  = positions where valid_mask is True
    scoreable positions  = positions where valid_mask[t] and valid_mask[t+1]
                           (a next-token target exists)

Keeping B = 1 is intentional: it makes the (sequence, position) -> column mapping
trivial to verify and costs nothing at 2048 tokens on a T4.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence

import torch


@dataclass
class CalibBatch:
    """One calibration sequence and everything aligned to it."""

    input_ids: torch.Tensor                 # [1, T] int64
    attention_mask: torch.Tensor            # [1, T] int64
    valid_mask: torch.Tensor                # [T] bool
    weights: Optional[torch.Tensor] = None  # [T] float32, None => uniform
    meta: Dict[str, object] = field(default_factory=dict)

    @property
    def length(self) -> int:
        return int(self.input_ids.shape[1])

    @property
    def n_valid(self) -> int:
        return int(self.valid_mask.sum())

    def to(self, device) -> "CalibBatch":
        return CalibBatch(
            input_ids=self.input_ids.to(device),
            attention_mask=self.attention_mask.to(device),
            valid_mask=self.valid_mask,
            weights=self.weights,
            meta=self.meta,
        )

    def scoreable_positions(self) -> torch.Tensor:
        """Indices t where both t and t+1 are real tokens (a target exists)."""
        valid = self.valid_mask
        if valid.numel() < 2:
            return torch.zeros(0, dtype=torch.long)
        pair = valid[:-1] & valid[1:]
        return torch.nonzero(pair, as_tuple=False).flatten()


def target_positions(valid_mask: torch.Tensor) -> torch.Tensor:
    """Position indices whose *next* token is a real token."""
    if valid_mask.numel() < 2:
        return torch.zeros(0, dtype=torch.long)
    return torch.nonzero(valid_mask[:-1] & valid_mask[1:], as_tuple=False).flatten()


def batches_from_records(
    records: Sequence[Dict],
    max_seq_len: int,
    weights_per_record: Optional[Sequence[torch.Tensor]] = None,
    device: Optional[torch.device] = None,
    drop_incomplete_target: bool = True,
) -> List[CalibBatch]:
    """
    Build CalibBatches from JSONL records that already carry `token_ids`.

    `records[i]["token_ids"]` must be the exact ids used for the forward pass.
    Passing ids (rather than re-tokenising) is what makes notebook 01 -> 06
    bit-reproducible.
    """
    batches: List[CalibBatch] = []
    for idx, rec in enumerate(records):
        ids = rec.get("token_ids")
        if ids is None:
            raise KeyError(
                f"Record {idx} has no 'token_ids'. Notebook 01 stores them; "
                "re-tokenising here would silently change the calibration set."
            )
        ids = list(ids)[:max_seq_len]
        if drop_incomplete_target and len(ids) < 2:
            continue
        t = len(ids)
        input_ids = torch.tensor([ids], dtype=torch.long)
        mask = torch.ones(t, dtype=torch.long)
        valid = torch.ones(t, dtype=torch.bool)

        w = None
        if weights_per_record is not None:
            w = torch.as_tensor(weights_per_record[idx], dtype=torch.float32).flatten()
            if w.numel() != t:
                raise ValueError(
                    f"Weight vector for record {idx} has length {w.numel()} "
                    f"but the sequence has {t} tokens."
                )

        batch = CalibBatch(
            input_ids=input_ids,
            attention_mask=mask,
            valid_mask=valid,
            weights=w,
            meta={
                "record_index": idx,
                "prompt_id": rec.get("prompt_id"),
                "source": rec.get("source"),
                "length": t,
            },
        )
        if device is not None:
            batch = batch.to(device)
        batches.append(batch)
    return batches


def total_columns(batches: Sequence[CalibBatch]) -> int:
    return sum(b.n_valid for b in batches)


def describe_batches(batches: Sequence[CalibBatch]) -> Dict[str, object]:
    lengths = [b.length for b in batches]
    n_valid = [b.n_valid for b in batches]
    has_w = all(b.weights is not None for b in batches)
    return {
        "n_sequences": len(batches),
        "total_tokens": int(sum(n_valid)),
        "length_min": min(lengths) if lengths else 0,
        "length_max": max(lengths) if lengths else 0,
        "length_mean": round(sum(lengths) / len(lengths), 1) if lengths else 0.0,
        "weights_present": has_w,
        "sources": sorted({b.meta.get("source") for b in batches if b.meta.get("source")}),
    }


def concat_weights(batches: Sequence[CalibBatch]) -> Optional[torch.Tensor]:
    """Stack the per-batch weight vectors into one global pool-ordered vector."""
    if not all(b.weights is not None for b in batches):
        return None
    return torch.cat([b.weights.flatten() for b in batches], dim=0)


def calibration_sufficiency(batches: Sequence[CalibBatch], max_in_features: int) -> Dict[str, object]:
    """
    Is there enough calibration data for the Hessian to be well conditioned?

    `H` is (d_in x d_in) built from N token columns. If N < d_in the Gram matrix is
    structurally rank-deficient, and the ridge damping is the only thing making the
    solve possible -- at which point the weighting is not doing what you think.
    As a rule of thumb N >= 2*d_in is comfortable, N >= d_in is marginal.

    Called loudly by notebooks 02 and 06 because short generation traces are the most
    likely way this project silently produces garbage.
    """
    n = total_columns(batches)
    d = int(max_in_features)
    ratio = n / max(d, 1)
    if ratio >= 2.0:
        verdict, action = "ok", "proceed"
    elif ratio >= 1.0:
        verdict, action = "marginal", ("increase n_reasoning or gen_max_new_tokens "
                                       "before trusting the result")
    else:
        verdict, action = "rank-deficient", ("STOP: fewer tokens than input channels. "
                                             "Increase n_reasoning / gen_max_new_tokens "
                                             "or the Hessian will be dominated by damping")
    return {
        "total_tokens": int(n),
        "max_in_features": d,
        "tokens_per_channel": round(ratio, 2),
        "verdict": verdict,
        "action": action,
    }
'''
MODULES['calibration_data.py'] = r'''"""
Prompt sources, the mixed calibration corpus, and the contamination audit.

Rules enforced here (v2 §5.1, §9.4)
----------------------------------
1. Calibration prompts come from **training** splits only, and evaluation prompt ids
   are disjoint by construction.
2. The corpus is stored as records carrying `token_ids`, so every later stage uses
   byte-identical tokenisation.
3. An n-gram overlap audit is run against the evaluation benchmarks and written to
   disk. A non-zero overlap is reported, never silently ignored.
"""
from __future__ import annotations

import hashlib
import re
from typing import Dict, List, Optional, Sequence, Tuple

from .config import CalibConfig

PROMPT_TEMPLATES = {
    "plain": "{question}",
    "instruct": "Solve the following math problem step by step.\n\n{question}",
    "chat": ("<|im_start|>user\nSolve the following problem step by step. "
             "Put the final answer in \\boxed{{}}.\n\n{question}<|im_end|>\n<|im_start|>assistant\n"),
}


def stable_id(text: str, prefix: str = "") -> str:
    return prefix + hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


def load_prompt_source(cfg: CalibConfig, limit: Optional[int] = None) -> List[Dict]:
    """
    Load training-split reasoning prompts.

    Uses `datasets` (available on Kaggle with internet enabled). For an offline run,
    substitute a local JSONL via `load_local_prompts`.
    """
    from datasets import load_dataset

    ds = load_dataset(cfg.prompt_dataset, cfg.prompt_config, split=cfg.prompt_split)
    rows: List[Dict] = []
    for i, ex in enumerate(ds):
        if limit is not None and i >= limit:
            break
        q = ex.get(cfg.prompt_field)
        if not q:
            continue
        q = str(q).strip()
        rows.append({
            "prompt_id": stable_id(q, prefix=cfg.prompt_split + ":"),
            "prompt": q,
            "source": cfg.prompt_dataset,
            "source_split": cfg.prompt_split,
        })
    return rows


def load_local_prompts(path: str, field: str = "question") -> List[Dict]:
    """Offline fallback: JSONL with one {field, ...} object per line."""
    import json

    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            q = str(obj.get(field, "")).strip()
            if q:
                rows.append({
                    "prompt_id": obj.get("prompt_id") or stable_id(q, "local:"),
                    "prompt": q,
                    "source": "local",
                    "source_split": "local",
                })
    return rows


def load_general_anchors(cfg: CalibConfig, limit: Optional[int] = None) -> List[str]:
    """C4 / WikiText-2 raw text used as general-language anchors (25% of the corpus)."""
    from datasets import load_dataset

    ds = load_dataset(cfg.general_dataset, cfg.general_config, split=cfg.general_split)
    texts: List[str] = []
    buf: List[str] = []
    for ex in ds:
        t = str(ex.get(cfg.general_field, "")).strip()
        if not t:
            continue
        buf.append(t)
        joined = "\n".join(buf)
        if len(joined) > 4000:
            texts.append(joined)
            buf = []
            if limit is not None and len(texts) >= limit:
                break
    if buf and (limit is None or len(texts) < limit):
        texts.append("\n".join(buf))
    return texts[:limit] if limit else texts


def sample_prompts(prompts: Sequence[Dict], n: int, seed: int = 0,
                   exclude_ids: Optional[set] = None) -> List[Dict]:
    """Deterministic, reproducible sample with optional id exclusion."""
    import random

    pool = [p for p in prompts if not exclude_ids or p["prompt_id"] not in exclude_ids]
    rng = random.Random(seed)
    if n >= len(pool):
        return list(pool)
    return rng.sample(pool, n)


def render_prompt(question: str, template: str = "plain") -> str:
    tpl = PROMPT_TEMPLATES.get(template, PROMPT_TEMPLATES["plain"])
    return tpl.format(question=question)


def normalise_for_overlap(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def ngrams(text: str, n: int) -> List[str]:
    words = normalise_for_overlap(text).split(" ")
    if len(words) < n:
        return []
    return [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]


def leak_audit(calib_texts: Sequence[str], eval_texts: Sequence[str], n: int = 13
               ) -> Dict[str, object]:
    """
    n-gram overlap between calibration prompts/traces and evaluation prompts.

    Returns the fraction of calibration n-grams found in the evaluation set, plus
    example collisions so they can be inspected rather than trusted.
    """
    calib_ng = set()
    for t in calib_texts:
        calib_ng.update(ngrams(t, n))
    eval_ng = set()
    for t in eval_texts:
        eval_ng.update(ngrams(t, n))

    if not calib_ng:
        return {"n": n, "calib_ngrams": 0, "eval_ngrams": len(eval_ng),
                "overlap_count": 0, "overlap_frac": 0.0, "examples": []}

    inter = calib_ng & eval_ng
    examples = sorted(inter)[:10]
    return {
        "n": n,
        "calib_ngrams": len(calib_ng),
        "eval_ngrams": len(eval_ng),
        "overlap_count": len(inter),
        "overlap_frac": len(inter) / len(calib_ng),
        "examples": examples,
        "clean": len(inter) == 0,
    }


def build_calibration_plan(cfg: CalibConfig, eval_prompt_ids: Optional[set] = None
                           ) -> Dict[str, object]:
    """
    Decide *which* prompts and anchors to use, before any generation happens.

    Returning a plan (instead of generating inside this module) keeps the expensive
    step in notebook 01 and makes the plan inspectable/printable.
    """
    prompts = load_prompt_source(cfg, limit=max(64, cfg.n_reasoning * 4))
    chosen = sample_prompts(prompts, cfg.n_reasoning, seed=cfg.seed,
                            exclude_ids=eval_prompt_ids)
    anchors = load_general_anchors(cfg, limit=cfg.n_general)
    return {
        "reasoning_prompts": chosen,
        "general_anchors": anchors,
        "prompt_source": cfg.prompt_dataset,
        "prompt_split": cfg.prompt_split,
        "seed": cfg.seed,
        "n_reasoning": len(chosen),
        "n_general": len(anchors),
        "prompt_ids": [p["prompt_id"] for p in chosen],
    }


CALIB_EVAL_PROTOCOL = {
    "mix": "75% self-generated reasoning traces, 25% general anchors (v1 §2.2)",
    "reasoning_source": "training splits only (GSM8K/MATH train)",
    "general_source": "WikiText-2 raw train",
    "decoding": "greedy (deterministic) for the primary corpus",
    "leak_audit": "13-gram overlap vs every evaluation set, reported in results",
}
'''
MODULES['config.py'] = r'''"""
Typed configuration objects for the EB-SW-GPTAQ proxy study.

Every notebook loads exactly one YAML file per component so that no experiment
parameter is hard-coded in a script (Milestone 0 requirement, v2 §7).
"""
from __future__ import annotations

import dataclasses
import os
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import yaml

# --------------------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------------------


@dataclass
class ModelConfig:
    """Which model, in which precision, on which device."""

    model_id: str = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    revision: Optional[str] = None          # pin a commit hash for reproducibility
    dtype: str = "float16"                  # "float16" | "bfloat16" | "float32"
    fp32_lm_head: bool = True               # T4 has no native BF16 -> keep head in FP32
    device: str = "cuda"
    attn_implementation: str = "sdpa"       # "sdpa" | "eager"
    trust_remote_code: bool = False
    max_seq_len: int = 2048
    exclude_name_patterns: Tuple[str, ...] = ("lm_head",)   # never quantized


# --------------------------------------------------------------------------------------
# Calibration corpus
# --------------------------------------------------------------------------------------


@dataclass
class CalibConfig:
    """Mixed calibration corpus: self-generated reasoning traces + general anchors."""

    n_reasoning: int = 20                   # pilot size; v2 §5.1 allows 16-32
    n_general: int = 7
    n_holdout: int = 8                      # disjoint held-out prompts for the proxy gate
    max_seq_len: int = 2048
    seed: int = 0
    min_trace_tokens: int = 64              # drop degenerate/truncated traces

    prompt_dataset: str = "openai/gsm8k"
    prompt_config: str = "main"
    prompt_split: str = "train"
    prompt_field: str = "question"

    general_dataset: str = "wikitext"
    general_config: str = "wikitext-2-raw-v1"
    general_split: str = "train"
    general_field: str = "text"

    gen_max_new_tokens: int = 1280
    gen_do_sample: bool = False             # deterministic primary corpus (v2 §5.1)
    gen_temperature: float = 0.6
    gen_top_p: float = 0.95

    ngram_leak_n: int = 13                  # contamination audit granularity


# --------------------------------------------------------------------------------------
# Quantization
# --------------------------------------------------------------------------------------


@dataclass
class QuantConfig:
    """One quantization run. Everything a reader needs to reproduce it."""

    bits: int = 4
    group_size: int = 128
    sym: bool = False                       # False => asymmetric grid with zero-point
    damp_percent: float = 0.01
    act_order: bool = False                 # Hessian-diagonal channel permutation
    true_sequential: bool = False           # exact greedy OBC instead of lazy blocks
    blocksize: int = 128
    working_dtype: str = "float32"          # dtype of the solver's working copy of W

    # Hessian / cross-covariance accumulation
    hessian_dtype: str = "float32"          # "float32" | "float64"
    min_tokens_per_layer: int = 256         # skip layers with too few calibration columns

    # asymmetric reconstruction (GPTAQ)
    asym: bool = True                       # False => weighted symmetric GPTQ
    drift_correction: bool = True           # True => capture X~ from the FP16 model
    #                                        False => X~ := X (plain weighted GPTQ)

    # grid fitting
    mse_clip: bool = False
    n_clip_grid: int = 20

    # module selection
    exclude_name_patterns: Tuple[str, ...] = ("lm_head",)
    min_in_features: int = 16

    # diagnostics
    diag_max_dim: int = 2560                # full eigendecomposition below this size
    diag_topk: int = 64                     # randomised sketch size above it
    diag_every_n: int = 2                   # only measure 1 layer in N (n=8960 is costly)


# --------------------------------------------------------------------------------------
# Token weighting (the signal under test)
# --------------------------------------------------------------------------------------


@dataclass
class WeightConfig:
    """
    Map per-position sensitivity scores to calibration weights.

    signal:
        "none"    -> all weights exactly 1.0 (uniform GPTQ/GPTAQ baseline)
        "jsd"     -> same-prefix JSD between FP16 and a quantized probe
        "entropy" -> FP16 output entropy (open-loop secondary signal)
        "kl"      -> same-prefix forward KL (heavy-tailed negative control)
        "random"  -> random weights with the same distributional shape
    """

    signal: str = "jsd"
    probe: str = "rtn"                      # which probe produced the JSD scores
    clip_quantile: float = 0.99             # upper winsorization, 0.0 disables
    standardize: bool = True                # z-score after clipping
    normalize: str = "global"               # "global" | "per_sequence"

    # Either a fixed inverse temperature beta, or a target ESS floor (bisection).
    beta: Optional[float] = None
    ess_floor: Optional[float] = 0.5
    beta_max: float = 50.0

    sink_mask: bool = True                  # neutralise BOS/attention-sink positions
    sink_positions: Tuple[int, ...] = (0,)
    shuffle_within_sequence: bool = False   # specificity control (breaks alignment)
    seed: int = 0


# --------------------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------------------


@dataclass
class EvalConfig:
    n_holdout: int = 64                     # v2 §5.4 / v1 §3.4 use 64 dev problems
    max_seq_len: int = 2048
    seed: int = 0
    chunk_positions: int = 64               # positions per JSD/KL chunk (vocab memory)
    topk: int = 0                           # 0 => exact full-vocabulary JSD
    high_entropy_frac: float = 0.10         # top-10% fork positions
    bootstrap: int = 10000


# --------------------------------------------------------------------------------------
# YAML plumbing
# --------------------------------------------------------------------------------------


def _coerce(value: Any) -> Any:
    """Turn YAML lists into tuples where the dataclass declares a Tuple field."""
    if isinstance(value, list):
        return tuple(value)
    return value


def load_yaml(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as fh:
        return yaml.safe_load(fh) or {}


def from_dict(cls, data: Dict[str, Any]):
    """Build a dataclass from a dict, ignoring unknown keys, coercing lists to tuples."""
    if data is None:
        return cls()
    fields = {f.name: f for f in dataclasses.fields(cls)}
    kwargs = {}
    for key, value in data.items():
        if key in fields:
            kwargs[key] = _coerce(value)
    return cls(**kwargs)


def load_config(path: str, cls):
    """Load `path` into dataclass `cls`. Returns cls() if path is None."""
    if path is None:
        return cls()
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Config file not found: {path}\n"
            f"Expected one of the files shipped in the repo's configs/ directory."
        )
    return from_dict(cls, load_yaml(path))


def config_to_dict(cfg) -> Dict[str, Any]:
    return dataclasses.asdict(cfg)
'''
MODULES['evaluation.py'] = r'''"""
Held-out evaluation: teacher-forced fidelity, strata, and (later) generation metrics.

The proxy gate is decided by teacher-forced fidelity on identical prefixes, because
that isolates the quantiser's effect on the output distribution from decoding noise.
Generation-based accuracy is a *downstream* pilot and only runs after the gate
(v2 §5.4, §6).

Memory note
-----------
`teacher_forced_fidelity` expects both the FP16 reference and the quantized model to
be resident. For a 1.5B model in FP16 that is ~6 GB, which fits a T4. For 4B+ use the
`swap` mode (not implemented by default) or evaluate in two passes while caching only
the positions you need.
"""
from __future__ import annotations

import math
import re
from typing import Dict, List, Optional, Sequence, Tuple

import torch

from .batching import CalibBatch, target_positions
from .jsd import divergence_per_position, log_softmax_fp32


# ======================================================================================
# Teacher-forced fidelity
# ======================================================================================


@torch.no_grad()
def teacher_forced_fidelity(
    ref_wrapper,
    q_wrapper,
    batches: Sequence[CalibBatch],
    chunk_positions: int = 64,
    topk: int = 0,
    keep_per_position: bool = True,
) -> Dict[str, object]:
    """
    Per-sequence KL(FP16 || quantized), JSD, and teacher-forced NLL for both models.

    Returns
    -------
    dict with
        per_sequence : list of {mean_kl, mean_jsd, mean_entropy, nll_ref, nll_q, n}
        arrays       : concatenated per-position scores (optional, for strata)
        aggregate    : mean/summary values used in the results table
    """
    per_sequence: List[Dict[str, float]] = []
    all_kl: List[torch.Tensor] = []
    all_jsd: List[torch.Tensor] = []
    all_ent: List[torch.Tensor] = []
    all_seq_id: List[torch.Tensor] = []
    total_nll_ref = 0.0
    total_nll_q = 0.0
    total_n = 0

    for si, batch in enumerate(batches):
        pos = target_positions(batch.valid_mask)
        if pos.numel() == 0:
            continue
        pos = pos.to(batch.input_ids.device)

        logits_ref, _ = ref_wrapper.forward_logits(batch.input_ids, batch.attention_mask)
        logits_q, _ = q_wrapper.forward_logits(batch.input_ids, batch.attention_mask)
        lr = logits_ref[0].index_select(0, pos)          # [P, V]
        lq = logits_q[0].index_select(0, pos)

        kl = divergence_per_position(lr, lq, "kl", chunk=chunk_positions)
        jsd = divergence_per_position(lr, lq, "jsd", chunk=chunk_positions, topk=topk)
        ent = divergence_per_position(lr, lr, "entropy", chunk=chunk_positions)

        labels = batch.input_ids[0].index_select(0, pos + 1)
        lp_ref = log_softmax_fp32(lr).gather(1, labels.unsqueeze(1)).squeeze(1)
        lp_q = log_softmax_fp32(lq).gather(1, labels.unsqueeze(1)).squeeze(1)

        per_sequence.append({
            "seq_index": si,
            "n_positions": int(pos.numel()),
            "mean_kl": float(kl.mean()),
            "mean_jsd": float(jsd.mean()),
            "mean_entropy": float(ent.mean()),
            "nll_ref": float(-lp_ref.mean()),
            "nll_q": float(-lp_q.mean()),
            "ppl_ref": float(torch.exp(-lp_ref.mean())),
            "ppl_q": float(torch.exp(-lp_q.mean())),
        })
        total_nll_ref += float(-lp_ref.sum())
        total_nll_q += float(-lp_q.sum())
        total_n += int(pos.numel())

        if keep_per_position:
            all_kl.append(kl.cpu())
            all_jsd.append(jsd.cpu())
            all_ent.append(ent.cpu())
            all_seq_id.append(torch.full((pos.numel(),), si, dtype=torch.long))

    if not per_sequence:
        return {"per_sequence": [], "aggregate": {"n_sequences": 0}}

    n = len(per_sequence)
    aggregate = {
        "n_sequences": n,
        "n_positions": total_n,
        "mean_kl": sum(r["mean_kl"] for r in per_sequence) / n,
        "mean_jsd": sum(r["mean_jsd"] for r in per_sequence) / n,
        "mean_entropy": sum(r["mean_entropy"] for r in per_sequence) / n,
        "nll_ref": total_nll_ref / max(total_n, 1),
        "nll_q": total_nll_q / max(total_n, 1),
        "ppl_ref": math.exp(min(total_nll_ref / max(total_n, 1), 60)),
        "ppl_q": math.exp(min(total_nll_q / max(total_n, 1), 60)),
        "nll_delta": (total_nll_q - total_nll_ref) / max(total_n, 1),
    }

    out: Dict[str, object] = {"per_sequence": per_sequence, "aggregate": aggregate}
    if keep_per_position:
        out["arrays"] = {
            "kl": torch.cat(all_kl),
            "jsd": torch.cat(all_jsd),
            "entropy": torch.cat(all_ent),
            "seq_id": torch.cat(all_seq_id),
        }
    return out


def strata_report(
    arrays: Dict[str, torch.Tensor],
    thresholds: Dict[str, float],
    prefix: str = "",
) -> Dict[str, object]:
    """
    Break fidelity down into high-entropy fork positions, commitment positions and
    the rest, using thresholds frozen on *calibration* scores (no leakage).

    `thresholds` must come from the calibration split, e.g.
        {"entropy_p90": 4.31, "jsd_p90": 0.021}
    """
    ent = arrays["entropy"]
    jsd = arrays["jsd"]
    kl = arrays["kl"]

    report: Dict[str, object] = {}
    if "entropy_p90" in thresholds:
        hi = ent >= thresholds["entropy_p90"]
        report["high_entropy"] = {
            "n": int(hi.sum()),
            "mean_kl": float(kl[hi].mean()) if hi.any() else None,
            "mean_jsd": float(jsd[hi].mean()) if hi.any() else None,
        }
        report["low_entropy"] = {
            "n": int((~hi).sum()),
            "mean_kl": float(kl[~hi].mean()) if (~hi).any() else None,
            "mean_jsd": float(jsd[~hi].mean()) if (~hi).any() else None,
        }
        if hi.any() and (~hi).any():
            report["kl_ratio_high_over_low"] = (
                float(kl[hi].mean()) / max(float(kl[~hi].mean()), 1e-12)
            )
    if "jsd_p90" in thresholds:
        hi = jsd >= thresholds["jsd_p90"]
        report["high_jsd"] = {"n": int(hi.sum()), "mean_kl": float(kl[hi].mean()) if hi.any() else None}
        report["low_jsd"] = {"n": int((~hi).sum()), "mean_kl": float(kl[~hi].mean()) if (~hi).any() else None}
    report["overall"] = {
        "n": int(ent.numel()),
        "mean_kl": float(kl.mean()),
        "mean_jsd": float(jsd.mean()),
        "mean_entropy": float(ent.mean()),
    }
    if prefix:
        report = {f"{prefix}{k}": v for k, v in report.items()}
    return report


# ======================================================================================
# Generation-based pilot metrics (notebook 09)
# ======================================================================================

_BOXED = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
_NUMBER = re.compile(r"-?\d+(?:\.\d+)?")


def extract_boxed(text: str) -> Optional[str]:
    matches = _BOXED.findall(text or "")
    return matches[-1].strip() if matches else None


def extract_final_answer(text: str) -> Optional[str]:
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed
    nums = _NUMBER.findall(text or "")
    return nums[-1] if nums else None


def normalise_answer(ans: Optional[str]) -> str:
    if ans is None:
        return ""
    a = ans.strip().lower()
    a = a.replace("$", "").replace("\\%", "%").replace(",", "")
    a = re.sub(r"\s+", "", a)
    a = a.rstrip(".")
    return a


def answers_match(pred: Optional[str], gold: Optional[str]) -> bool:
    """Cheap numeric/symbolic match. Upgrade to math-verify for the full phase."""
    p, g = normalise_answer(pred), normalise_answer(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    try:
        return abs(float(p) - float(g)) < 1e-6
    except Exception:
        return p.replace(" ", "") == g.replace(" ", "")


def answer_appeared_early(trace: str, gold: Optional[str], tail_tokens: int = 120) -> bool:
    """
    Overthinking detector: the correct answer appears in the trace, but the model
    keeps going and does not commit to it (v1 §3.4 failure taxonomy, category a).
    """
    g = normalise_answer(gold if gold is not None else extract_final_answer(trace))
    if not g:
        return False
    text = normalise_answer(trace)
    first = text.find(g)
    if first < 0:
        return False
    # is the answer present well before the end of the trace?
    return first < max(0, len(text) - tail_tokens)


@torch.no_grad()
def generate_and_score(
    wrapper,
    records: Sequence[Dict],
    max_new_tokens: int = 4096,
    do_sample: bool = False,
    temperature: float = 0.6,
    top_p: float = 0.95,
    template: str = "plain",
    limit: Optional[int] = None,
) -> Dict[str, object]:
    """
    Generate one answer per problem and compute Acc / mean CoT length / TPCA.

    TPCA (tokens per correct answer) is the deployment-relevant metric: it charges a
    method for both the accuracy it loses and the extra tokens it spends.
    """
    from .calibration_data import render_prompt

    tokenizer = wrapper.tokenizer
    rows: List[Dict] = []
    items = list(records)[:limit] if limit else list(records)

    for item in items:
        rendered = item.get("rendered_prompt") or render_prompt(item["prompt"], template)
        enc = tokenizer(rendered, return_tensors="pt").to(wrapper.device)
        plen = int(enc["input_ids"].shape[1])
        out = wrapper.generate(
            enc["input_ids"], attention_mask=enc["attention_mask"],
            max_new_tokens=int(max_new_tokens), do_sample=bool(do_sample),
            temperature=float(temperature), top_p=float(top_p),
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
        new_ids = out[0][plen:].tolist()
        text = tokenizer.decode(new_ids, skip_special_tokens=True)

        gold = item.get("answer")
        pred = extract_final_answer(text)
        correct = answers_match(pred, gold)
        rows.append({
            "prompt_id": item.get("prompt_id"),
            "n_tokens": len(new_ids),
            "hit_max_new_tokens": len(new_ids) >= int(max_new_tokens),
            "pred": pred,
            "gold": gold,
            "correct": bool(correct),
            "overthinking": bool(answer_appeared_early(text, gold)) and not correct,
            "trace_head": text[:400],
        })

    n = len(rows)
    n_correct = sum(1 for r in rows if r["correct"])
    total_tokens = sum(r["n_tokens"] for r in rows)
    return {
        "n": n,
        "accuracy": n_correct / max(n, 1),
        "n_correct": n_correct,
        "mean_cot_tokens": total_tokens / max(n, 1),
        "tpca": total_tokens / max(n_correct, 1),
        "truncation_rate": sum(1 for r in rows if r["hit_max_new_tokens"]) / max(n, 1),
        "overthinking_rate": sum(1 for r in rows if r["overthinking"]) / max(n, 1),
        "rows": rows,
        "per_item_correct": [bool(r["correct"]) for r in rows],
        "per_item_tokens": [int(r["n_tokens"]) for r in rows],
    }
'''
MODULES['geometry.py'] = r'''"""
Activation-geometry diagnostics (hypothesis H2 of the pivot).

Question: do high-JSD positions occupy a *different* activation subspace from
low-JSD positions, or are they merely the same directions with larger magnitude?

If the answer is "same subspace", then column weighting is just a rescaling of
directions that uniform calibration already sees, and the geometry-based story
collapses. That is exactly the kind of pre-registered falsification the pivot asks
for, so these diagnostics are reported for several layers rather than the best one.

All routines take [d, N] activation matrices (columns are tokens) and return plain
Python numbers so they can be JSON-serialised straight into the results file.
"""
from __future__ import annotations

import math
from typing import Dict, List, Optional, Sequence

import torch

from .quant_core import spectral_summary


def weighted_covariance(X: torch.Tensor, w: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Covariance-style Gram matrix of X [d, N], optionally token-weighted."""
    Xf = X.detach().to(torch.float32)
    if w is not None:
        wv = torch.as_tensor(w, dtype=torch.float32, device=Xf.device).flatten().clamp_min(0)
        Xf = Xf * wv.sqrt().unsqueeze(0)
    n = max(Xf.shape[1], 1)
    return (Xf @ Xf.transpose(0, 1)) / n


def top_subspace(H: torch.Tensor, k: int = 16, seed: int = 0) -> torch.Tensor:
    """Top-k principal directions (orthonormal, [d, k]) of a PSD matrix."""
    d = H.shape[0]
    k = max(1, min(k, d))
    Hs = H.detach().to(torch.float32).cpu()
    if d <= 1024:
        evals, evecs = torch.linalg.eigh(Hs)
        return evecs[:, -k:].flip(1)
    torch.manual_seed(int(seed))
    _U, _S, V = torch.svd_lowrank(Hs, q=min(k + 8, d), niter=4)
    return V[:, -k:].flip(1)


def principal_angles(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """
    Principal angles (radians, ascending) between the column spaces of A and B.

    Computed from the singular values of Q_A^T Q_B, which is numerically stable and
    avoids forming the projector matrices.
    """
    Qa = torch.linalg.qr(A.to(torch.float32).cpu(), mode="reduced").Q
    Qb = torch.linalg.qr(B.to(torch.float32).cpu(), mode="reduced").Q
    s = torch.linalg.svdvals(Qa.transpose(0, 1) @ Qb).clamp(-1.0, 1.0)
    return torch.acos(s).sort().values


def subspace_overlap(A: torch.Tensor, B: torch.Tensor) -> Dict[str, float]:
    """
    Mean cos^2 of principal angles: 1.0 = identical subspace, ~k/d = chance.

    `chance` is the expected overlap for two random k-dim subspaces in d dims,
    so the comparison to report is overlap vs chance, not overlap vs 1.
    """
    d = A.shape[0]
    k = min(A.shape[1], B.shape[1])
    angles = principal_angles(A, B)
    cos2 = torch.cos(angles).pow(2)
    return {
        "k": int(k),
        "d": int(d),
        "mean_cos2": float(cos2.mean()),
        "min_cos2": float(cos2.min()),
        "max_cos2": float(cos2.max()),
        "mean_angle_deg": float(math.degrees(float(angles.mean()))),
        "max_angle_deg": float(math.degrees(float(angles.max()))),
        "chance_cos2": float(min(1.0, k / max(d, 1))),
    }


def split_by_threshold(scores: torch.Tensor, threshold: float, valid: Optional[torch.Tensor] = None):
    """Boolean masks for high / low score groups, honouring a validity mask."""
    s = scores.detach().to(torch.float32).flatten()
    ok = torch.isfinite(s)
    if valid is not None:
        ok &= valid.flatten().bool()
    high = ok & (s >= threshold)
    low = ok & (s < threshold)
    return high, low


def matched_random_split(n: int, n_high: int, seed: int = 0) -> torch.Tensor:
    """Random subset of size n_high -- the matched-size control for H2."""
    g = torch.Generator().manual_seed(int(seed))
    perm = torch.randperm(n, generator=g)
    mask = torch.zeros(n, dtype=torch.bool)
    mask[perm[:n_high]] = True
    return mask


def geometry_report(
    X: torch.Tensor,
    high_mask: torch.Tensor,
    low_mask: torch.Tensor,
    k: int = 16,
    max_dim: int = 2560,
    seed: int = 0,
) -> Dict[str, object]:
    """
    Full H2 diagnostic block for one (layer, module, probe) triple.

    Reports the spectrum of each group, the high/low subspace overlap, and the
    overlap of the high group with a matched random subset of the same size (the
    control that decides whether any difference is real).
    """
    Xf = X.detach().to(torch.float32)
    hi = Xf[:, high_mask.bool()]
    lo = Xf[:, low_mask.bool()]
    n_hi, n_lo = hi.shape[1], lo.shape[1]
    if n_hi < k + 4 or n_lo < k + 4:
        return {"error": f"too few columns (high={n_hi}, low={n_lo}) for k={k}"}

    H_hi = weighted_covariance(hi)
    H_lo = weighted_covariance(lo)
    A = top_subspace(H_hi, k, seed)
    B = top_subspace(H_lo, k, seed)

    rand_mask = matched_random_split(Xf.shape[1], n_hi, seed=seed)
    X_rand = Xf[:, rand_mask]
    A_rand = top_subspace(weighted_covariance(X_rand), k, seed=seed)

    return {
        "n_high": int(n_hi),
        "n_low": int(n_lo),
        "k": int(k),
        "high_vs_low": subspace_overlap(A, B),
        "high_vs_random": subspace_overlap(A, A_rand),
        "low_vs_random": subspace_overlap(B, A_rand),
        "spectrum_high": spectral_summary(H_hi, max_dim=max_dim),
        "spectrum_low": spectral_summary(H_lo, max_dim=max_dim),
        "norm_high_median": float(hi.norm(dim=0).median()),
        "norm_low_median": float(lo.norm(dim=0).median()),
    }


def aggregate_geometry(reports: List[Dict]) -> Dict[str, object]:
    """Aggregate across layers/modules; never cherry-pick a single favourable layer."""
    ok = [r for r in reports if "high_vs_low" in r]
    if not ok:
        return {"n_reports": 0}
    mean_ov = sum(r["high_vs_low"]["mean_cos2"] for r in ok) / len(ok)
    mean_ctrl = sum(r["high_vs_random"]["mean_cos2"] for r in ok) / len(ok)
    return {
        "n_reports": len(ok),
        "mean_overlap_high_vs_low": mean_ov,
        "mean_overlap_high_vs_random": mean_ctrl,
        "mean_gap_over_control": mean_ov - mean_ctrl,
        "frac_layers_below_control": sum(
            1 for r in ok if r["high_vs_low"]["mean_cos2"] < r["high_vs_random"]["mean_cos2"]
        ) / len(ok),
        "mean_angle_deg": sum(r["high_vs_low"]["mean_angle_deg"] for r in ok) / len(ok),
    }
'''
MODULES['jsd.py'] = r'''"""
Same-prefix distributional divergence in FP32.

Everything here operates on *logits at aligned positions*: row t of a logits tensor
predicts token t+1. The notebooks always pass a matching set of positions, which is
produced by `batching.target_positions`.

Numerics (v2 §9.1)
------------------
- log_softmax is always computed in FP32 (`logits.float()` first), because FP16
  softmax over a 152k vocabulary loses the tail that JSD is supposed to measure.
- log M is formed with `logaddexp(logP, logQ) - log 2` instead of averaging
  probabilities, which keeps subnormal tail probabilities representable.
- Probabilities are only materialised per chunk of positions.

JSD is bounded in [0, ln 2] with natural logs, symmetric, and exactly zero for
identical distributions -- all three are asserted by the unit tests in notebook 04.
"""
from __future__ import annotations

from typing import Dict, Optional, Tuple

import torch

LOG2 = 0.6931471805599453


def log_softmax_fp32(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """FP32 log-softmax over the last dimension (vocabulary)."""
    x = logits.to(torch.float32)
    if temperature != 1.0:
        x = x / float(temperature)
    return torch.log_softmax(x, dim=-1)


def _jsd_from_logprobs(log_p: torch.Tensor, log_q: torch.Tensor) -> torch.Tensor:
    """JSD(P||Q) for matching [N, V] log-probabilities. Returns [N] in nats."""
    log_m = torch.logaddexp(log_p, log_q) - LOG2
    p = log_p.exp()
    q = log_q.exp()
    kl_pm = (p * (log_p - log_m)).sum(dim=-1)
    kl_qm = (q * (log_q - log_m)).sum(dim=-1)
    return (0.5 * (kl_pm + kl_qm)).clamp_min(0.0)


def _kl_from_logprobs(log_p: torch.Tensor, log_q: torch.Tensor) -> torch.Tensor:
    """Forward KL(P||Q). Unbounded: a heavy-tailed negative control (v2 §4.1)."""
    p = log_p.exp()
    return (p * (log_p - log_q)).sum(dim=-1).clamp_min(0.0)


def _entropy_from_logprobs(log_p: torch.Tensor) -> torch.Tensor:
    p = log_p.exp()
    return -(p * log_p).sum(dim=-1)


def _gather_topk(logits: torch.Tensor, k: int) -> torch.Tensor:
    return torch.topk(logits, k=min(k, logits.shape[-1]), dim=-1).indices


def _jsd_topk(log_p: torch.Tensor, log_q: torch.Tensor, k: int) -> torch.Tensor:
    """
    Top-k JSD with the residual tail kept as an explicit extra bin.

    The union of both top-k index sets is retained, the probability mass outside that
    union is collapsed into one residual bin per distribution, and JSD is computed on
    the compact (<= 2k+1)-bin distribution. Collapsing the tail instead of dropping it
    is what keeps the approximation from being biased downward (v2 §9.1).
    """
    p = log_p.exp()
    q = log_q.exp()
    idx = torch.cat([_gather_topk(p, k), _gather_topk(q, k)], dim=-1)
    keep = torch.zeros_like(p).scatter_(1, idx, 1.0)
    p_m = p * keep
    q_m = q * keep
    p_res = (1.0 - p_m.sum(dim=-1, keepdim=True)).clamp_min(0.0)
    q_res = (1.0 - q_m.sum(dim=-1, keepdim=True)).clamp_min(0.0)
    p2 = torch.cat([p_m, p_res], dim=-1)
    q2 = torch.cat([q_m, q_res], dim=-1)
    m = 0.5 * (p2 + q2)
    log_m = m.clamp_min(1e-30).log()
    log_p2 = p2.clamp_min(1e-30).log()
    log_q2 = q2.clamp_min(1e-30).log()
    kl_pm = torch.where(p2 > 0, p2 * (log_p2 - log_m), torch.zeros_like(p2)).sum(dim=-1)
    kl_qm = torch.where(q2 > 0, q2 * (log_q2 - log_m), torch.zeros_like(q2)).sum(dim=-1)
    return (0.5 * (kl_pm + kl_qm)).clamp_min(0.0)


def divergence_per_position(
    logits_ref: torch.Tensor,
    logits_probe: torch.Tensor,
    kind: str = "jsd",
    chunk: int = 64,
    topk: int = 0,
    temperature: float = 1.0,
) -> torch.Tensor:
    """
    Per-position divergence between two aligned logits tensors.

    Parameters
    ----------
    logits_ref, logits_probe : [T, V] or [B, T, V], same shape, FP16/BF16/FP32.
    kind : "jsd" | "kl" | "entropy"   ("entropy" ignores logits_probe)
    chunk : positions processed at once (vocabulary memory control)
    topk : 0 = exact full-vocabulary; >0 = top-k union approximation

    Returns a tensor with the leading shape of the inputs.
    """
    if logits_ref.shape != logits_probe.shape and kind != "entropy":
        raise ValueError(f"shape mismatch: {tuple(logits_ref.shape)} vs "
                         f"{tuple(logits_probe.shape)}")
    lead = logits_ref.shape[:-1]
    v = logits_ref.shape[-1]
    lr = logits_ref.reshape(-1, v)
    lp = logits_probe.reshape(-1, v)
    out = torch.zeros(lr.shape[0], dtype=torch.float32, device=lr.device)

    for i in range(0, lr.shape[0], chunk):
        j = min(i + chunk, lr.shape[0])
        a = log_softmax_fp32(lr[i:j], temperature)
        if kind == "entropy":
            out[i:j] = _entropy_from_logprobs(a)
            continue
        b = log_softmax_fp32(lp[i:j], temperature)
        if topk and topk > 0:
            out[i:j] = _jsd_topk(a, b, topk)
        elif kind == "jsd":
            out[i:j] = _jsd_from_logprobs(a, b)
        elif kind == "kl":
            out[i:j] = _kl_from_logprobs(a, b)
        else:
            raise ValueError(f"unknown divergence kind: {kind}")
    return out.reshape(lead)


def jsd_summary(scores: torch.Tensor) -> Dict[str, float]:
    """Distribution summary used by the results table."""
    s = scores.detach().to(torch.float32).flatten()
    if s.numel() == 0:
        return {}
    qs = torch.quantile(s, torch.tensor([0.5, 0.9, 0.99], dtype=torch.float32))
    return {
        "n": int(s.numel()),
        "mean": float(s.mean()),
        "std": float(s.std(unbiased=False)),
        "median": float(qs[0]),
        "p90": float(qs[1]),
        "p99": float(qs[2]),
        "max": float(s.max()),
        "frac_above_half_ln2": float((s > 0.5 * LOG2).float().mean()),
    }


def assert_jsd_sanity(v: int = 512, device: str = "cpu", seed: int = 0) -> Dict[str, float]:
    """
    Unit checks for the JSD implementation. Called by notebook 04 before the real run.

    Verified properties: symmetry, non-negativity, upper bound ln 2, zero for
    identical distributions, and top-k approximation error against exact JSD.
    """
    g = torch.Generator(device="cpu").manual_seed(seed)
    x = torch.randn(4, v, generator=g)
    y = torch.randn(4, v, generator=g)
    d_xy = divergence_per_position(x, y, "jsd", chunk=2)
    d_yx = divergence_per_position(y, x, "jsd", chunk=2)
    d_xx = divergence_per_position(x, x, "jsd", chunk=2)
    d_topk = divergence_per_position(x, y, "jsd", chunk=2, topk=32)
    return {
        "symmetry_max_abs_diff": float((d_xy - d_yx).abs().max()),
        "self_jsd_max": float(d_xx.max()),
        "min_jsd": float(d_xy.min()),
        "max_jsd": float(d_xy.max()),
        "ln2": LOG2,
        "within_bound": bool(d_xy.max() <= LOG2 + 1e-5),
        "topk_mean_abs_err": float((d_topk - d_xy).abs().mean()),
    }


def assert_causal_alignment(tol: float = 1e-6) -> Dict[str, object]:
    """
    Verify that logit row t is scored against token t+1 on a controlled tensor.

    This is the guard for the single most dangerous silent bug in the project
    (v2 §9.2). It uses a deterministic toy construction, not a model, so it costs
    nothing and can run before any GPU time is spent.

    Construction: token ids = [1..8]; "logits" whose value equals the row index.
    After a causal shift, row t must pair with id t+2 for t in 0..6.
    """
    ids = torch.arange(1, 9, dtype=torch.long).unsqueeze(0)          # [1, 8]
    scores = torch.arange(7, dtype=torch.float32).reshape(1, 7, 1)   # [1, 7, 1]

    # The canonical shift used everywhere in this repo:
    shift_logits = scores[:, :-1, :]          # rows 0..6  (drops the last position)
    shift_labels = ids[:, 1:]                 # ids 1..7  (drops the first token)

    pairs = list(zip(shift_logits.reshape(-1).tolist(), shift_labels.reshape(-1).tolist()))
    expected = [(float(t), t + 2) for t in range(7)]
    matches = [abs(a - c) < tol and b == d
               for (a, b), (c, d) in zip(pairs, expected)]

    return {
        "n_pairs": len(pairs),
        "all_pairs_correct": bool(all(matches)),
        "first_pair": pairs[0],
        "last_pair": pairs[-1],
        "note": "logits[t] must be scored against token_ids[t+1]",
    }
'''
MODULES['logging_utils.py'] = r'''"""
Artifact paths, IO helpers, and the reproducibility manifest.

Kaggle reality this module solves
---------------------------------
Every Kaggle notebook runs on a fresh VM. `/kaggle/working` does not survive into
another notebook unless that notebook's output is attached as an input, in which
case it appears read-only under `/kaggle/input/<slug>/`.

So this module keeps two roots:
  write_root()  -> always writable  (/kaggle/working/artifacts, or $EBSW_ARTIFACT_ROOT)
  find(rel)     -> searches write_root() first, then every /kaggle/input/*/artifacts

`require(rel, produced_by=...)` raises an error that names the notebook you must run
first, which is the whole point: no silent empty-directory bugs.
"""
from __future__ import annotations

import glob
import hashlib
import json
import os
import platform
import random
import subprocess
import sys
import time
from typing import Any, Dict, List, Optional

# --------------------------------------------------------------------------------------
# Roots
# --------------------------------------------------------------------------------------

WORK_ROOT = "/kaggle/working"


def write_root(subdir: str = "") -> str:
    """Writable artifact root. Created on demand."""
    base = os.environ.get("EBSW_ARTIFACT_ROOT")
    if not base:
        if os.path.isdir("/kaggle/working"):
            base = os.path.join(WORK_ROOT, "artifacts")
        else:
            # Local fallback so the library is importable off-Kaggle.
            base = os.path.join(os.getcwd(), "artifacts")
    path = os.path.join(base, subdir) if subdir else base
    os.makedirs(path, exist_ok=True)
    return path


def search_roots() -> List[str]:
    """All places `find` looks, in priority order."""
    roots = [write_root()]
    env = os.environ.get("EBSW_EXTRA_ROOTS")
    if env:
        roots.extend(p for p in env.split(os.pathsep) if p)
    roots.extend(sorted(glob.glob("/kaggle/input/*/artifacts")))
    return roots


def find(rel: str) -> Optional[str]:
    """Locate an artifact by relative path across all search roots."""
    for root in search_roots():
        candidate = os.path.join(root, rel)
        if os.path.exists(candidate):
            return candidate
    return None


def require(rel: str, produced_by: str) -> str:
    """Locate an artifact or fail loudly, naming the notebook that produces it."""
    path = find(rel)
    if path is None:
        raise FileNotFoundError(
            f"Missing required artifact: {rel}\n"
            f"  Produced by notebook: {produced_by}\n"
            f"  Searched: {search_roots()}\n"
            f"Fix: run {produced_by} first and save its version, then attach that\n"
            f"     version's output as an input to this notebook (Add Input ->\n"
            f"     Notebook Output), or set EBSW_ARTIFACT_ROOT to a directory\n"
            f"     containing it."
        )
    return path


def out(rel: str) -> str:
    """Path under the writable root; parent directories created."""
    path = os.path.join(write_root(), rel)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    return path


# --------------------------------------------------------------------------------------
# IO
# --------------------------------------------------------------------------------------


def save_json(obj: Any, rel: str) -> str:
    path = out(rel)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=_json_default)
    return path


def load_json(rel_or_path: str, produced_by: str = "the notebook that creates it") -> Any:
    path = rel_or_path if os.path.isabs(rel_or_path) else require(rel_or_path, produced_by)
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


def append_jsonl(obj: Any, rel: str) -> str:
    path = out(rel)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(obj, default=_json_default) + "\n")
    return path


def read_jsonl(rel_or_path: str, produced_by: str = "the notebook that creates it") -> List[Dict]:
    path = rel_or_path if os.path.isabs(rel_or_path) else require(rel_or_path, produced_by)
    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(rows: List[Dict], rel: str) -> str:
    path = out(rel)
    with open(path, "w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, default=_json_default) + "\n")
    return path


def _json_default(obj: Any):
    try:
        import numpy as np

        if isinstance(obj, np.generic):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
    except Exception:
        pass
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return str(obj)


# --------------------------------------------------------------------------------------
# Run log (one JSONL line per stage, appended by every notebook)
# --------------------------------------------------------------------------------------


def log_event(stage: str, **fields) -> str:
    record = {"ts": time.strftime("%Y-%m-%dT%H:%M:%S"), "stage": stage}
    record.update(fields)
    return append_jsonl(record, "logs/run_log.jsonl")


# --------------------------------------------------------------------------------------
# Reproducibility manifest (Milestone 0)
# --------------------------------------------------------------------------------------


def _pkg_version(name: str) -> Optional[str]:
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return None


def environment_manifest(extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Everything a reviewer needs to know to reproduce this run."""
    manifest: Dict[str, Any] = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "packages": {
            name: _pkg_version(name)
            for name in ("torch", "transformers", "datasets", "tokenizers", "accelerate",
                         "numpy", "scipy", "safetensors", "sentencepiece", "yaml")
        },
    }
    try:
        import torch

        manifest["torch"] = {
            "version": torch.__version__,
            "cuda": torch.version.cuda,
            "cuda_available": torch.cuda.is_available(),
            "device_count": torch.cuda.device_count(),
            "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "capability": [torch.cuda.get_device_capability(i)
                           for i in range(torch.cuda.device_count())],
            "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        }
    except Exception as exc:  # pragma: no cover
        manifest["torch"] = {"error": str(exc)}

    try:
        manifest["git"] = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5
        ).stdout.strip() or None
    except Exception:
        manifest["git"] = None

    if extra:
        manifest.update(extra)
    return manifest


# --------------------------------------------------------------------------------------
# Seeding
# --------------------------------------------------------------------------------------


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except Exception:
        pass
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


# --------------------------------------------------------------------------------------
# Small utilities
# --------------------------------------------------------------------------------------


def human_bytes(n: float) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024.0:
            return f"{n:3.1f}{unit}"
        n /= 1024.0
    return f"{n:.1f}PB"


def file_sha256(path: str, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            block = fh.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def gpu_memory_report() -> Dict[str, Any]:
    try:
        import torch

        if not torch.cuda.is_available():
            return {"cuda": False}
        free, total = torch.cuda.mem_get_info()
        return {
            "cuda": True,
            "allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 3),
            "reserved_gb": round(torch.cuda.memory_reserved() / 2**30, 3),
            "free_gb": round(free / 2**30, 3),
            "total_gb": round(total / 2**30, 3),
        }
    except Exception as exc:  # pragma: no cover
        return {"cuda": False, "error": str(exc)}


def banner(text: str, width: int = 88) -> None:
    print("=" * width)
    print(text)
    print("=" * width)


def section(text: str, width: int = 88) -> None:
    print(f"\n--- {text} " + "-" * max(0, width - len(text) - 5))


def zip_artifacts(zip_name: str, rel_paths: Optional[List[str]] = None) -> str:
    """
    Zip selected artifacts so they can be re-attached as a Kaggle dataset.

    This is the escape hatch when attaching notebook outputs directly is awkward:
    download the zip, create a private dataset from it, attach that.
    """
    import zipfile

    root = write_root()
    target = os.path.join(WORK_ROOT, zip_name) if os.path.isdir(WORK_ROOT) else zip_name
    paths = rel_paths
    if paths is None:
        paths = []
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                full = os.path.join(dirpath, fn)
                paths.append(os.path.relpath(full, root))
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as zf:
        for rel in paths:
            full = os.path.join(root, rel)
            if os.path.exists(full):
                zf.write(full, os.path.join("artifacts", rel))
    return target
'''
MODULES['model_loader.py'] = r'''"""
Model loading and the forward-pass wrapper used by every notebook.

Why a wrapper
-------------
T4 has no native BF16 tensor cores and Qwen-family logits can exceed the FP16
range (|logit| > 65504 is unusual but the *pre-softmax* accumulation can overflow
in fp16 for some Qwen2 checkpoints). Two guards are applied here:

1. `fp32_lm_head=True` keeps `lm_head` in FP32 and computes logits as
   `lm_head(hidden.float())`, so the vocabulary projection never overflows.
2. `safe_logits()` asserts finite values and reports the offending batch instead
   of letting NaNs propagate silently into JSD/KL (where they become `nan`
   scores that quietly destroy the weighting).

The wrapper also owns the layer/linear enumeration that the quantizer walks, so
there is exactly one definition of "which modules get quantized".
"""
from __future__ import annotations

import time
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn

from .config import ModelConfig


DTYPES = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}


class ModelWrapper:
    """Thin, explicit wrapper around a HF causal LM."""

    def __init__(self, model: nn.Module, tokenizer, cfg: ModelConfig):
        self.model = model
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.device = next(model.parameters()).device
        self._lm_head_fp32 = None
        self._decoder_layers = None

    # ------------------------------------------------------------------ basics

    @property
    def dtype(self) -> torch.dtype:
        return next(self.model.parameters()).dtype

    @property
    def n_layers(self) -> int:
        return len(self.decoder_layers())

    def _backbone_attr(self) -> str:
        for attr in ("model", "transformer", "decoder"):
            if hasattr(self.model, attr):
                return attr
        raise AttributeError(
            "Cannot find the transformer backbone on this model. "
            "Expected .model / .transformer / .decoder."
        )

    def transformer(self) -> nn.Module:
        """The backbone without the LM head."""
        return getattr(self.model, self._backbone_attr())

    def decoder_layers(self) -> List[nn.Module]:
        if self._decoder_layers is not None:
            return self._decoder_layers
        backbone = self.transformer()
        for attr in ("layers", "h", "block", "blocks"):
            if hasattr(backbone, attr):
                self._decoder_layers = list(getattr(backbone, attr))
                return self._decoder_layers
        raise AttributeError("Cannot find transformer blocks (layers/h/block).")

    def linear_modules(self, prefix: str = "") -> "Dict[str, nn.Linear]":
        """
        Every nn.Linear in the whole model, keyed by a name that is resolvable via
        `model.get_submodule(name)` (so it is prefixed with the backbone attribute).
        The LM head and too-small layers are excluded.
        """
        out: Dict[str, nn.Linear] = {}
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear):
                continue
            if any(pat in name for pat in self.cfg.exclude_name_patterns if pat):
                continue
            out[f"{prefix}{name}"] = mod
        return out

    def layer_linears(self, layer_idx: int) -> Dict[str, nn.Linear]:
        prefix = self._layer_prefix(layer_idx)
        found: Dict[str, nn.Linear] = {}
        for name, mod in self.decoder_layers()[layer_idx].named_modules():
            if not isinstance(mod, nn.Linear):
                continue
            full = f"{prefix}.{name}" if name else prefix
            if any(pat in full for pat in self.cfg.exclude_name_patterns if pat):
                continue
            found[full] = mod
        return found

    def _layer_prefix(self, layer_idx: int) -> str:
        backbone_attr = self._backbone_attr()
        backbone = self.transformer()
        for attr in ("layers", "h", "block", "blocks"):
            if hasattr(backbone, attr):
                return f"{backbone_attr}.{attr}.{layer_idx}"
        return f"{backbone_attr}.{layer_idx}"

    # -------------------------------------------------------------- lm head

    def lm_head_weight(self) -> torch.Tensor:
        head = self.model.get_output_embeddings()
        if self.cfg.fp32_lm_head:
            if self._lm_head_fp32 is None:
                self._lm_head_fp32 = head.weight.detach().to(torch.float32)
            return self._lm_head_fp32
        return head.weight.detach()

    # -------------------------------------------------------------- forward

    @torch.no_grad()
    def forward_logits(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        need_hidden: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, ...]]]:
        """
        Run the backbone and produce FP32 logits.

        Returns (logits [B, T, V] float32, hidden_states or None).

        The head is applied manually so that (a) FP16 overflow cannot happen in
        the vocabulary projection on a T4, and (b) activations can be captured
        from the backbone alone without materialising logits.
        """
        out = self.transformer()(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=need_hidden,
            return_dict=True,
        )
        hidden = out.last_hidden_state
        logits = self._project(hidden)
        hs = getattr(out, "hidden_states", None)
        return logits, hs

    def _project(self, hidden: torch.Tensor) -> torch.Tensor:
        """
        Hidden states -> FP32 logits, chunked over positions.

        Hidden states are [.., d]; the output is [.., V]. Chunking means a
        [T=2048, V=151936] float32 tensor (1.2 GB) never exists at once.
        """
        weight = self.lm_head_weight()
        lead = hidden.shape[:-1]
        flat = hidden.reshape(-1, hidden.shape[-1])
        chunks = []
        step = 256
        for i in range(0, flat.shape[0], step):
            piece = flat[i:i + step].to(torch.float32)
            chunks.append(piece @ weight.to(piece.device).T)
        logits = torch.cat(chunks, dim=0) if len(chunks) > 1 else chunks[0]
        return logits.reshape(*lead, logits.shape[-1])

    @torch.no_grad()
    def forward_hidden(self, input_ids: torch.Tensor,
                       attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Backbone forward *without* the vocabulary projection.

        Activation capture and divergence extraction use this: projecting 32k
        calibration tokens through a 152k-way head costs more FLOPs than the whole
        transformer pass and produces a tensor nobody needs.
        """
        out = self.transformer()(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        return out.last_hidden_state

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, **kwargs) -> torch.Tensor:
        return self.model.generate(input_ids=input_ids, **kwargs)

    # -------------------------------------------------------------- weights

    def set_weight(self, name: str, value: torch.Tensor) -> None:
        mod = self.model.get_submodule(name)
        mod.weight.data.copy_(value.to(mod.weight.dtype).to(mod.weight.device))

    def get_weight(self, name: str) -> torch.Tensor:
        return self.model.get_submodule(name).weight.detach()

    def named_linears(self) -> Dict[str, nn.Linear]:
        return self.linear_modules()


def safe_logits(logits: torch.Tensor, tag: str = "") -> torch.Tensor:
    """Fail loudly on non-finite logits instead of propagating NaNs."""
    if not torch.isfinite(logits).all():
        n_bad = int((~torch.isfinite(logits)).sum())
        raise FloatingPointError(
            f"Non-finite logits detected {tag}: {n_bad} values. "
            "If this is a Qwen checkpoint running in fp16 on a T4, keep "
            "fp32_lm_head=True (the default) and consider attn_implementation='eager'."
        )
    return logits


def load_model(
    cfg: ModelConfig,
    load_in_dtype: Optional[torch.dtype] = None,
    verbose: bool = True,
) -> ModelWrapper:
    """Load tokenizer + causal LM in the requested dtype."""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    dtype = load_in_dtype or DTYPES.get(cfg.dtype, torch.float16)
    t0 = time.time()

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_id, revision=cfg.revision, trust_remote_code=cfg.trust_remote_code
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    common = dict(
        revision=cfg.revision,
        device_map=cfg.device if cfg.device != "cpu" else "cpu",
        attn_implementation=cfg.attn_implementation,
        trust_remote_code=cfg.trust_remote_code,
        low_cpu_mem_usage=True,
    )
    try:
        model = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=dtype, **common)
    except TypeError:
        # transformers 5.x renamed torch_dtype -> dtype
        model = AutoModelForCausalLM.from_pretrained(cfg.model_id, dtype=dtype, **common)
    model.eval()
    model.config.use_cache = False

    wrapper = ModelWrapper(model, tokenizer, cfg)
    if verbose:
        n_par = sum(p.numel() for p in model.parameters())
        print(f"Loaded {cfg.model_id}  dtype={dtype}  params={n_par/1e9:.3f}B  "
              f"layers={wrapper.n_layers}  in {time.time()-t0:.1f}s")
        print(f"  revision={getattr(model.config, '_name_or_path', cfg.model_id)} "
              f"lm_head_dtype={wrapper.lm_head_weight().dtype}")
        if torch.cuda.is_available():
            print(f"  cuda_allocated={torch.cuda.memory_allocated()/2**30:.2f} GB")
    return wrapper


def model_fingerprint(cfg: ModelConfig, tokenizer) -> Dict[str, object]:
    """Model/tokenizer revision record for the results table (v2 §11)."""
    return {
        "model_id": cfg.model_id,
        "requested_revision": cfg.revision,
        "dtype": cfg.dtype,
        "vocab_size": getattr(tokenizer, "vocab_size", None),
        "tokenizer_name": getattr(tokenizer, "name_or_path", None),
        "pad_token_id": getattr(tokenizer, "pad_token_id", None),
        "eos_token_id": getattr(tokenizer, "eos_token_id", None),
    }
'''
MODULES['probe_quantization.py'] = r'''"""
Preliminary quantized probes used to define the JSD signal.

The probe answers one question: "where does quantization actually perturb the next
token distribution?" It must therefore be *cheap and fixed* -- not tuned, not
strong. The pivot's probe-swap analysis (v2 §5.3) exists precisely because the
signal might be an artefact of the probe, so the probe is a parameter, never a
constant.

Two probes:
    rtn  - round-to-nearest, no data, ~seconds. Zero Hessian, zero activation passes.
    gptq - uniform-weight GPTQ on the same calibration set (drift correction off for
           cost, documented as a deviation from full GPTAQ).

Both are produced by the same solver used for the variants, so the probe cannot
accidentally be "better" because of a different quantiser implementation.
"""
from __future__ import annotations

import time
from typing import Dict, List, Optional, Sequence

import torch

from .config import QuantConfig
from .logging_utils import log_event, save_json
from . import quant_core


def probe_config(bits: int = 4, group_size: int = 128, sym: bool = False,
                 method: str = "rtn") -> QuantConfig:
    """Fixed probe hyper-parameters. Deliberately plain: no act_order, no clipping."""
    return QuantConfig(
        bits=bits,
        group_size=group_size,
        sym=sym,
        damp_percent=0.01,
        act_order=False,
        true_sequential=False,
        asym=False,                # symmetric reconstruction for the probe
        drift_correction=False,    # one pass per layer -> cheap
        mse_clip=False,
        hessian_dtype="float32",
    )


def build_probe(wrapped, batches: Optional[Sequence] = None, method: str = "rtn",
                bits: int = 4, group_size: int = 128, sym: bool = False,
                verbose: bool = True, tag: Optional[str] = None) -> Dict[str, object]:
    """
    Build and apply a probe model in place. Returns metadata + results dict.

    `batches` is required for method="gptq" and ignored for "rtn".
    """
    cfg = probe_config(bits=bits, group_size=group_size, sym=sym, method=method)
    t0 = time.time()

    if method == "rtn":
        results = quant_core.quantize_model_rtn(wrapped, cfg, verbose=verbose)
    elif method in ("gptq", "gptaq"):
        if batches is None:
            raise ValueError("method='gptq' needs calibration batches")
        cfg.asym = method == "gptaq"
        results = quant_core.quantize_model(wrapped, batches, cfg, verbose=verbose)
    else:
        raise ValueError(f"unknown probe method: {method}")

    tag = tag or f"probe_{method}_w{bits}"
    summary = quant_core.summarize_results(results)
    meta = {
        "tag": tag,
        "method": method,
        "bits": bits,
        "group_size": group_size,
        "sym": sym,
        "drift_correction": cfg.drift_correction,
        "asym": cfg.asym,
        "n_modules": summary["n_modules"],
        "rel_fro_err_by_module": summary["rel_fro_err_by_module"],
        "wall_clock_s": round(time.time() - t0, 1),
        "summary": summary,
    }
    if verbose:
        print(f"probe built: {tag}  modules={summary['n_modules']}  "
              f"{meta['wall_clock_s']}s")
    log_event("build_probe", **{k: v for k, v in meta.items() if k != "summary"})
    return {"results": results, "meta": meta}


def save_probe(results: Dict, meta: Dict, rel_dir: str) -> str:
    """Persist a probe so notebooks 04/08 do not have to re-quantise."""
    from .logging_utils import out

    path = out(f"{rel_dir}/probe.pt")
    quant_core.save_quant_state(results, path, meta=meta)
    save_json(meta, f"{rel_dir}/probe_meta.json")
    return path


def load_probe(wrapped, rel_dir: str, apply: bool = True) -> Dict:
    """Load and (optionally) apply a saved probe to the current model."""
    from .logging_utils import require

    path = require(f"{rel_dir}/probe.pt", produced_by="03_build_probe.ipynb")
    return quant_core.load_quant_state(wrapped, path, apply=apply)


def describe_probe(meta: Dict) -> str:
    return (f"probe={meta['method']} W{meta['bits']}A16 g{meta['group_size']} "
            f"sym={meta['sym']} drift={meta['drift_correction']} "
            f"modules={meta['n_modules']}")


# ======================================================================================
# Milestone 3: same-prefix score extraction
# ======================================================================================


@torch.no_grad()
def extract_position_scores(
    ref_wrapper,
    probe_wrapper,
    batches,
    chunk: int = 64,
    topk: int = 0,
    with_topk_validation: bool = False,
    verbose: bool = True,
) -> Dict[str, object]:
    """
    Per-position JSD / forward-KL / entropy between the FP16 reference and a probe.

    Memory-safe by construction: the backbone forward is done once per sequence to get
    hidden states ([T, d], a few MB), and the vocabulary projection happens one chunk
    of positions at a time, so a [T, V] float32 logits tensor is never materialised.

    Alignment contract (v2 §9.1-9.2):
        only positions t with a valid target t+1 are scored, and the returned score
        vector for a sequence has exactly one entry per scored position, with the
        matching position indices in `positions`. Nothing is silently zero-padded.

    Returns
    -------
    dict with per-sequence lists of tensors plus a `summary` block.
    """
    from .batching import target_positions
    from .jsd import divergence_per_position, jsd_summary

    seq_jsd: List[torch.Tensor] = []
    seq_ent: List[torch.Tensor] = []
    seq_kl: List[torch.Tensor] = []
    seq_pos: List[torch.Tensor] = []
    seq_ids: List[torch.Tensor] = []
    topk_err: List[float] = []

    for bi, batch in enumerate(batches):
        pos = target_positions(batch.valid_mask)
        if pos.numel() == 0:
            continue
        h_ref = ref_wrapper.forward_hidden(batch.input_ids, batch.attention_mask)[0]
        h_probe = probe_wrapper.forward_hidden(batch.input_ids, batch.attention_mask)[0]

        jsd = torch.zeros(pos.numel(), dtype=torch.float32, device=h_ref.device)
        ent = torch.zeros_like(jsd)
        kl = torch.zeros_like(jsd)

        for i in range(0, pos.numel(), chunk):
            idx = pos[i:i + chunk]
            lr = ref_wrapper._project(h_ref.index_select(0, idx))
            lq = probe_wrapper._project(h_probe.index_select(0, idx))
            jsd[i:i + idx.numel()] = divergence_per_position(lr, lq, "jsd", chunk=chunk)
            ent[i:i + idx.numel()] = divergence_per_position(lr, lr, "entropy", chunk=chunk)
            kl[i:i + idx.numel()] = divergence_per_position(lr, lq, "kl", chunk=chunk)
            if with_topk_validation and topk and bi < 1:
                approx = divergence_per_position(lr, lq, "jsd", chunk=chunk, topk=topk)
                topk_err.append(float(
                    (approx - jsd[i:i + idx.numel()]).abs().mean()
                    / jsd[i:i + idx.numel()].mean().clamp_min(1e-12)
                ))

        seq_jsd.append(jsd.cpu())
        seq_ent.append(ent.cpu())
        seq_kl.append(kl.cpu())
        seq_pos.append(pos.cpu())
        seq_ids.append(torch.full((pos.numel(),), bi, dtype=torch.long))
        if verbose and (bi + 1) % 4 == 0:
            print(f"    scored {bi+1}/{len(batches)} sequences")

    if not seq_jsd:
        return {"jsd": [], "entropy": [], "kl": [], "summary": {"n_positions": 0}}

    all_jsd = torch.cat(seq_jsd)
    summary = {
        "n_sequences": len(seq_jsd),
        "n_positions": int(all_jsd.numel()),
        "jsd": jsd_summary(all_jsd),
        "entropy": jsd_summary(torch.cat(seq_ent)),
        "kl": jsd_summary(torch.cat(seq_kl)),
        "topk": int(topk),
        "topk_mean_rel_err": (sum(topk_err) / len(topk_err)) if topk_err else None,
        "chunk": int(chunk),
    }
    return {
        "jsd": seq_jsd, "entropy": seq_ent, "kl": seq_kl,
        "positions": seq_pos, "seq_id": seq_ids, "summary": summary,
    }
'''
MODULES['quant_core.py'] = r'''"""
Group-wise fake quantizer and the weighted asymmetric second-order solver.

This is the method under test. Everything else in the repository exists to
measure whether it helps.

What the solver optimises
-------------------------
For a linear layer with pretrained weight W in R^{out x in}, calibration inputs
X in R^{in x N} (from the *partially quantised* model), reference inputs
X~ in R^{in x N} (from the FP16 model), and a positive diagonal token weight
matrix D = diag(w_1..w_N) with mean(w) = 1:

    min_{W_hat in Q_b}  1/2 * || (W_hat X - W X~) D^{1/2} ||_F^2
                        + lam/2 * || W_hat - W ||_F^2

With DW = W_hat - W and DX = X~ - X this is exactly

    1/2 tr(DW H_w DW^T) - tr(DW (W C_w)^T) + const
    H_w = X D X^T + lam*I          (accumulated in FP32/FP64)
    C_w = (X~ - X) D X^T           (raw cross-covariance, no ridge inside)

The unconstrained optimum is DW* = W C_w H_w^{-1}. Because the quantiser still
constrains W_hat to the grid, we do not "shift then quantise". Instead we follow
GPTAQ: fit the grid (scale/zero) on the ORIGINAL W, then absorb the asymmetric
optimum into the greedy column-by-column OBC update. The identity used is
elementary but worth writing down, because it is what makes the code short:

    fixing column q to its quantised value, the optimal values of the remaining
    columns F are

        W[:, F] <- (W + P)[:, F] - ((W + P)[:, q] - q_hat) * Hinv[q, F] / Hinv[q, q]

    with P = W C_w Hinv. i.e. pre-shifting the *running target* by P and then
    applying the standard GPTQ rule is mathematically exact for the quadratic.
    (Verified in notebook 06 by the direct-objective unit test.)

Where the "weighted" and "asymmetric" parts live
------------------------------------------------
- weighting   : inside H_w and C_w (the accumulator applies sqrt(w) / w).
- asymmetry   : P, and it is switched off when Cw is None.

`asym=False` therefore gives weighted symmetric GPTQ (the EB-SW-GPTQ ablation) and
`Cw=None` gives a uniform-weight GPTAQ-class baseline.

Outputs
-------
`QuantizedLinear` carries the actual integer codes (q_int), the group parameters
(scales, zeros, g_idx) and the dequantised W_hat. `quantizer_adapter.py` turns the
first four into a packed-checkpoint layout without touching the solver.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn

from .config import QuantConfig


# ======================================================================================
# Group index bookkeeping
# ======================================================================================


def group_index(d_in: int, group_size: int) -> torch.Tensor:
    """Group id per input channel (GPTQ g_idx convention: groups along `in`)."""
    return (torch.arange(d_in, dtype=torch.long) // group_size).to(torch.int32)


def n_groups(d_in: int, group_size: int) -> int:
    return int(math.ceil(d_in / group_size))


# ======================================================================================
# Quantizer
# ======================================================================================


class GroupQuantizer:
    """
    Per-output-row, per-input-group uniform (or symmetric) fake quantiser.

    Kept deliberately free of any kernel/storage assumption: it returns integer
    codes plus group params, and a dequantised tensor. Real packing happens later.
    """

    def __init__(
        self,
        bits: int = 4,
        group_size: int = 128,
        sym: bool = False,
        mse_clip: bool = False,
        n_clip_grid: int = 20,
        eps: float = 1e-8,
    ):
        if not 2 <= bits <= 16:
            raise ValueError(f"bits must be in [2, 16], got {bits}")
        self.bits = int(bits)
        self.group_size = int(group_size)
        self.sym = bool(sym)
        self.mse_clip = bool(mse_clip)
        self.n_clip_grid = int(n_clip_grid)
        self.eps = float(eps)

        if self.sym:
            self.qmax = float(2 ** (self.bits - 1) - 1)
            self.qmin = -self.qmax
        else:
            self.qmax = float(2 ** self.bits - 1)
            self.qmin = 0.0
        self.code_dtype = torch.int16 if self.bits > 8 else torch.int8

    # ------------------------------------------------------------------ fitting

    def fit(self, W: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Fit group scale/zero on W [out, in]. Grid is fitted on the original W."""
        Wf = W.detach().to(torch.float32)
        out_f, in_f = Wf.shape
        g = self.group_size

        pad = (-in_f) % g
        if pad:
            # Repeating the last column cannot change any group's min/max.
            Wf = torch.cat([Wf, Wf[:, -1:].expand(out_f, pad)], dim=1)
        groups = Wf.reshape(out_f, -1, g)

        if self.sym:
            amax = groups.abs().amax(dim=-1)
            if self.mse_clip:
                scale = self._mse_clip_scale(groups, amax, sym=True)
            else:
                scale = amax / self.qmax
            scale = scale.clamp_min(self.eps)
            zero = torch.zeros_like(scale)
        else:
            lo = groups.amin(dim=-1)
            hi = groups.amax(dim=-1)
            if self.mse_clip:
                scale, zero = self._mse_clip_scale_asym(groups, hi)
            else:
                scale = ((hi - lo) / self.qmax).clamp_min(self.eps)
                zero = lo

        g_idx = group_index(in_f, g)
        return scale, zero, g_idx

    def _mse_clip_scale(self, groups: torch.Tensor, amax: torch.Tensor, sym: bool = True):
        """Pick the per-group clipping ratio in (0,1] minimising reconstruction MSE."""
        ratios = torch.linspace(1.0, 0.5, self.n_clip_grid, device=groups.device)
        best_err = None
        best_idx = torch.zeros_like(amax, dtype=torch.long)
        for i, r in enumerate(ratios):
            scale = (amax * r / self.qmax).clamp_min(self.eps)
            q = torch.round(groups / scale.unsqueeze(-1)).clamp(self.qmin, self.qmax)
            err = (groups - q * scale.unsqueeze(-1)).pow(2).mean(dim=-1)
            if best_err is None:
                best_err = err
            else:
                better = err < best_err
                best_err = torch.where(better, err, best_err)
                best_idx = torch.where(better, torch.full_like(best_idx, i), best_idx)
        return (amax * ratios[best_idx] / self.qmax).clamp_min(self.eps)

    def _mse_clip_scale_asym(self, groups: torch.Tensor, hi: torch.Tensor):
        lo = groups.amin(dim=-1)
        ratios = torch.linspace(1.0, 0.5, self.n_clip_grid, device=groups.device)
        best_err = None
        best_idx = torch.zeros_like(hi, dtype=torch.long)
        for i, r in enumerate(ratios):
            span = ((hi - lo) * r).clamp_min(self.eps)
            scale = span / self.qmax
            q = torch.round((groups - lo.unsqueeze(-1)) / scale.unsqueeze(-1)).clamp(
                self.qmin, self.qmax
            )
            err = (groups - (q * scale.unsqueeze(-1) + lo.unsqueeze(-1))).pow(2).mean(dim=-1)
            if best_err is None:
                best_err = err
            else:
                better = err < best_err
                best_err = torch.where(better, err, best_err)
                best_idx = torch.where(better, torch.full_like(best_idx, i), best_idx)
        scale = (((hi - lo).clamp_min(self.eps)) * ratios[best_idx] / self.qmax)
        return scale.clamp_min(self.eps), lo

    # ---------------------------------------------------------------- applying

    def expand_params(self, scale, zero, g_idx) -> Tuple[torch.Tensor, torch.Tensor]:
        """[out, G] group params -> [out, in] per-channel params."""
        return scale.index_select(1, g_idx.long()), zero.index_select(1, g_idx.long())

    def quantize_column(self, w: torch.Tensor, scale_col: torch.Tensor,
                        zero_col: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Quantise a single column w [out] with its per-row group params [out]."""
        if self.sym:
            q = torch.round(w / scale_col).clamp(self.qmin, self.qmax)
            dq = q * scale_col
        else:
            q = torch.round((w - zero_col) / scale_col).clamp(self.qmin, self.qmax)
            dq = q * scale_col + zero_col
        return q.to(self.code_dtype), dq

    def quantize(self, W: torch.Tensor, scale, zero, g_idx
                 ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Quantise a whole matrix. Returns (q_int [out,in], W_hat [out,in] fp32)."""
        Wf = W.detach().to(torch.float32)
        sc, zr = self.expand_params(scale, zero, g_idx)
        if self.sym:
            q = torch.round(Wf / sc).clamp(self.qmin, self.qmax)
            dq = q * sc
        else:
            q = torch.round((Wf - zr) / sc).clamp(self.qmin, self.qmax)
            dq = q * sc + zr
        return q.to(self.code_dtype), dq

    def dequantize(self, q_int: torch.Tensor, scale, zero, g_idx) -> torch.Tensor:
        sc, zr = self.expand_params(scale, zero, g_idx)
        q = q_int.to(torch.float32)
        return q * sc if self.sym else q * sc + zr


# ======================================================================================
# Streaming second-order statistics
# ======================================================================================


class WeightedHessian:
    """
    Streaming accumulator for

        H_w = sum_t w_t x_t x_t^T                      (in x in)
        C_w = sum_t w_t (x~_t - x_t) x_t^T             (in x in, optional)

    Accumulated in FP32 by default; FP64 is available but note that T4 FP64 is
    ~1/32 of FP32 throughput, so use FP64 only for small diagnostic runs.
    """

    def __init__(self, d_in: int, dtype: str = "float32", device=None,
                 track_cross: bool = True):
        self.d_in = int(d_in)
        self.dtype = {"float32": torch.float32, "float64": torch.float64}[dtype]
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.H = torch.zeros(d_in, d_in, dtype=self.dtype, device=self.device)
        self.Cw = (torch.zeros(d_in, d_in, dtype=self.dtype, device=self.device)
                   if track_cross else None)
        self.n_tokens = 0
        self.sum_w = 0.0
        self.sum_w2 = 0.0

    def add(self, X: torch.Tensor, X_ref: Optional[torch.Tensor] = None,
            w: Optional[torch.Tensor] = None) -> None:
        """X, X_ref: [d_in, T]. w: [T] positive token weights (mean-normalised)."""
        Xf = X.detach().to(device=self.device, dtype=self.dtype)
        if Xf.shape[0] != self.d_in:
            raise ValueError(f"expected {self.d_in} input channels, got {Xf.shape[0]}")
        t = Xf.shape[1]

        if w is None:
            wv = torch.ones(t, dtype=self.dtype, device=self.device)
        else:
            wv = torch.as_tensor(w, device=self.device, dtype=self.dtype).flatten()
            if wv.numel() != t:
                raise ValueError(f"weight length {wv.numel()} != {t} tokens")
            wv = wv.clamp_min(0.0)

        # H_w: scale columns by sqrt(w) then take the Gram matrix (v2 §9.3).
        Xw = Xf * wv.sqrt().unsqueeze(0)
        self.H.add_(Xw @ Xw.transpose(0, 1))

        # C_w uses the *unweighted* X on the right and w (not sqrt(w)) on the left.
        if self.Cw is not None and X_ref is not None:
            Xr = X_ref.detach().to(device=self.device, dtype=self.dtype)
            if Xr.shape != Xf.shape:
                raise ValueError(f"reference activation shape {tuple(Xr.shape)} "
                                 f"!= quantized shape {tuple(Xf.shape)}")
            diff = (Xr - Xf) * wv.unsqueeze(0)
            self.Cw.add_(diff @ Xf.transpose(0, 1))

        self.n_tokens += t
        self.sum_w += float(wv.sum())
        self.sum_w2 += float(wv.pow(2).sum())

    def ess_fraction(self) -> float:
        """Kish ESS / N over the accumulated pool."""
        if self.sum_w2 <= 0:
            return 0.0
        return float(self.sum_w ** 2 / self.sum_w2 / max(self.n_tokens, 1))

    def finalize(self) -> Dict[str, object]:
        return {
            "H": self.H,
            "Cw": self.Cw,
            "n_tokens": self.n_tokens,
            "ess_fraction": self.ess_fraction(),
            "dtype": str(self.dtype).replace("torch.", ""),
        }


# ======================================================================================
# Spectral diagnostics
# ======================================================================================


def _power_lambda_max(M: torch.Tensor, iters: int = 20) -> float:
    """Largest eigenvalue of a symmetric PSD matrix by power iteration (cheap: d²/iter)."""
    d = M.shape[0]
    v = torch.ones(d, dtype=torch.float32)
    v = v / v.norm()
    lam = 0.0
    for _ in range(iters):
        w = M @ v
        nrm = float(w.norm())
        if nrm < 1e-30:
            return 0.0
        v = w / nrm
        lam = nrm
    return lam


def spectral_summary(H: torch.Tensor, max_dim: int = 2560, topk: int = 64,
                     eps: float = 1e-12) -> Dict[str, float]:
    """
    Condition number / effective rank of a symmetric PSD matrix.

    Below `max_dim` this is an exact eigendecomposition. Above it, a full
    decomposition costs more than the rest of the layer's quantization (n=8960 is
    ~2.4e11 flops, i.e. ~2 minutes of CPU per call), so we use power iteration for
    lambda_max plus a q=`topk` randomised sketch for the effective rank, and report
    the condition number as None rather than inventing one.

    The *damped* condition number -- the one that actually matters for the solve -- is
    estimated after inversion and stored as `cond_damped_est`.
    """
    d = H.shape[0]
    out: Dict[str, float] = {"dim": d, "trace": float(torch.diagonal(H).sum())}
    Hs = H.detach().to(torch.float32).cpu()
    if d <= max_dim:
        ev = torch.linalg.eigvalsh(Hs).clamp_min(0.0)
        lam = ev.flip(0)                                # descending
        total = float(lam.sum())
        probs = (lam / total).clamp_min(eps) if total > 0 else lam
        ent = float(-(probs * probs.log()).sum())
        out.update({
            "lambda_max": float(lam[0]),
            "lambda_min": float(lam[-1]),
            "cond": float(lam[0] / max(float(lam[-1]), eps)),
            "eff_rank": float(torch.exp(torch.tensor(ent))),
            "eff_rank_norm": float(torch.exp(torch.tensor(ent)) / d),
            "n_below_1e-6_relmax": int((lam < 1e-6 * float(lam[0])).sum()),
            "partial": False,
        })
    else:
        lam_max = _power_lambda_max(Hs, iters=20)
        q = max(8, min(int(topk), d))
        try:
            _U, S, _V = torch.svd_lowrank(Hs, q=q, niter=2)
            lam_top = (S ** 2).clamp_min(0.0)
            share = lam_top / lam_top.sum().clamp_min(eps)
            eff = float(torch.exp(-(share.clamp_min(eps).log() * share).sum()))
            tail_mass = max(float(torch.diagonal(Hs).sum()) - float(lam_top.sum()), 0.0)
        except Exception:
            eff, tail_mass = float("nan"), float("nan")
        out.update({
            "lambda_max": float(lam_max),
            "lambda_min": None,
            "cond": None,
            "eff_rank_topk": eff,
            "eff_rank_topk_norm": eff / d if eff == eff else None,
            "residual_trace": tail_mass,
            "partial": True,
            "topk": q,
        })
    return out


# ======================================================================================
# Inverse
# ======================================================================================


def inverse_cholesky(H: torch.Tensor, jitters: Sequence[float] = (0.0, 1e-7, 1e-5, 1e-3, 1e-1)
                     ) -> Tuple[torch.Tensor, float]:
    """
    H^{-1} via Cholesky with escalating jitter, then an eigendecomposition fallback.

    Returns (Hinv, jitter_used). jitter_used == -1.0 means the eigen fallback ran.
    """
    d = H.shape[0]
    eye = torch.eye(d, dtype=H.dtype, device=H.device)
    for j in jitters:
        try:
            A = H if j == 0.0 else H + j * eye
            L = torch.linalg.cholesky(A)
            return torch.cholesky_inverse(L), float(j)
        except Exception:
            continue
    evals, evecs = torch.linalg.eigh(H)
    evals = evals.clamp_min(1e-8)
    return (evecs * (1.0 / evals)) @ evecs.transpose(0, 1), -1.0


# ======================================================================================
# Objective (used by the uniform-equivalence unit test)
# ======================================================================================


def reconstruction_objective(W_hat: torch.Tensor, W: torch.Tensor, H: torch.Tensor,
                             Cw: Optional[torch.Tensor] = None) -> float:
    """
    Evaluate 1/2 tr(DW H DW^T) - tr(DW (W Cw)^T) for DW = W_hat - W.

    H must be the *unridged* weighted Gram and Cw the raw weighted cross-covariance.
    This is the same quantity the solver minimises, so it is the ground truth for
    the "does the solver actually improve the objective" check.
    """
    DW = (W_hat.detach().to(torch.float32) - W.detach().to(torch.float32))
    Hf = H.detach().to(torch.float32)
    val = 0.5 * torch.sum(DW * (DW @ Hf))
    if Cw is not None:
        C = W.detach().to(torch.float32) @ Cw.detach().to(torch.float32)
        val = val - torch.sum(DW * C)
    return float(val)


# ======================================================================================
# The solver
# ======================================================================================


@dataclass
class QuantizedLinear:
    name: str
    q_int: torch.Tensor            # [out, in] integer codes (int8 / int16)
    scales: torch.Tensor           # [out, n_groups] fp32
    zeros: torch.Tensor            # [out, n_groups] fp32
    g_idx: torch.Tensor            # [in] int32
    W_hat: torch.Tensor            # dequantised, in the module's dtype
    meta: Dict[str, object] = field(default_factory=dict)

    def state_dict(self) -> Dict[str, object]:
        return {
            "name": self.name,
            "q_int": self.q_int.cpu(),
            "scales": self.scales.cpu(),
            "zeros": self.zeros.cpu(),
            "g_idx": self.g_idx.cpu(),
            "W_hat": self.W_hat.detach().cpu(),
            "meta": self.meta,
        }


def quantize_linear(
    name: str,
    W: torch.Tensor,
    H: Optional[torch.Tensor],
    Cw: Optional[torch.Tensor],
    cfg: QuantConfig,
    verbose: bool = False,
    compute_diagnostics: bool = True,
) -> Optional[QuantizedLinear]:
    """
    Quantise one linear layer.

    H is the FP32/FP64 weighted Gram (X D X^T). Pass H=None for RTN (no
    second-order solve). Cw is the weighted cross-covariance; pass None to disable
    the asymmetric term (weighted symmetric GPTQ).
    """
    t0 = time.time()
    W_true = W.detach().to(torch.float32).clone()          # pristine original
    out_f, in_f = W_true.shape
    if in_f < cfg.min_in_features:
        return None

    quant = GroupQuantizer(bits=cfg.bits, group_size=cfg.group_size, sym=cfg.sym,
                           mse_clip=cfg.mse_clip, n_clip_grid=cfg.n_clip_grid)

    # ---- 1. fit the grid on the ORIGINAL W (v1 §2.5: do not fit on the shifted matrix)
    scales, zeros, g_idx = quant.fit(W_true)

    # ---- 2. RTN shortcut: no Hessian, no sequential compensation
    if H is None:
        q_int, W_hat = quant.quantize(W_true, scales, zeros, g_idx)
        meta = {
            "method": "rtn", "bits": cfg.bits, "group_size": cfg.group_size,
            "sym": cfg.sym, "n_tokens": 0,
            "rel_fro_err": float((W_hat - W_true).norm() / W_true.norm().clamp_min(1e-12)),
            "time_s": round(time.time() - t0, 3),
        }
        return QuantizedLinear(name, q_int, scales, zeros, g_idx, W_hat.to(W.dtype), meta)

    # ---- 3. spectral diagnostics on the undamped Hessian (sampled: see diag_every_n)
    diag = (spectral_summary(H, cfg.diag_max_dim, cfg.diag_topk)
            if compute_diagnostics else {"skipped": True, "dim": in_f})

    Hd = H.detach().to(torch.float32).clone()
    dead = torch.diagonal(Hd) == 0
    n_dead = int(dead.sum())
    if n_dead:
        Hd[dead, dead] = 1.0
        W_solve = W_true.clone()
        W_solve[:, dead] = 0.0
    else:
        W_solve = W_true

    # ---- 4. adaptive ridge damping (v1 §2.5 step iv)
    trace_mean = float(torch.diagonal(Hd).mean())
    damp = float(cfg.damp_percent) * trace_mean
    if damp <= 0:
        damp = 1e-8 * max(1.0, float(torch.diagonal(Hd).abs().max()))
    idx = torch.arange(in_f, device=Hd.device)
    Hd[idx, idx] += damp
    Hinv, jitter_used = inverse_cholesky(Hd)

    # damped condition number -- the one that actually governs the solve. lambda_max(H)
    # by power iteration, lambda_max(H^-1) by power iteration: both are O(d²) per step,
    # so this is essentially free even at d=8960.
    if compute_diagnostics:
        try:
            lam_max_h = _power_lambda_max(Hd.detach().float().cpu(), iters=15)
            lam_max_hinv = _power_lambda_max(Hinv.detach().float().cpu(), iters=15)
            diag["cond_damped_est"] = float(lam_max_h * lam_max_hinv)
        except Exception:
            diag["cond_damped_est"] = None

    # ---- 5. asymmetric pre-shift P = W Cw Hinv  (exact quadratic optimum)
    use_asym = bool(cfg.asym) and (Cw is not None)
    if use_asym:
        P = (W_solve @ Cw.detach().to(torch.float32)) @ Hinv
    else:
        P = None

    # ---- 6. optional activation-order permutation (Hessian diagonal)
    if cfg.act_order:
        perm = torch.argsort(torch.diagonal(Hd), descending=True)
    else:
        perm = torch.arange(in_f, device=Hd.device)
    invperm = torch.argsort(perm)

    Hp = Hinv.index_select(0, perm).index_select(1, perm).contiguous()
    Wp = W_solve.index_select(1, perm).contiguous()
    Tmat = Wp if P is None else (Wp + P.index_select(1, perm))

    # per-(permuted)column group params, so the inner loop is a pure index
    groups_in_order = g_idx.long().index_select(0, perm)
    sc_cols = scales.index_select(1, groups_in_order)      # [out, in]
    zr_cols = zeros.index_select(1, groups_in_order)       # [out, in]

    Q = torch.zeros(out_f, in_f, dtype=torch.float32, device=W.device)
    Qint = torch.zeros(out_f, in_f, dtype=quant.code_dtype, device=W.device)

    # ---- 7. greedy OBC column updates
    if cfg.true_sequential:
        for j in range(in_f):
            w = Tmat[:, j]
            q_int, dq = quant.quantize_column(w, sc_cols[:, j], zr_cols[:, j])
            Q[:, j] = dq
            Qint[:, j] = q_int
            d = Hp[j, j]
            err = (w - dq) / d
            if j + 1 < in_f:
                Tmat[:, j + 1:] -= torch.outer(err, Hp[j, j + 1:])
    else:
        bs = max(1, int(cfg.blocksize))
        for i1 in range(0, in_f, bs):
            i2 = min(i1 + bs, in_f)
            cnt = i2 - i1
            W1 = Tmat[:, i1:i2].clone()
            Q1 = torch.zeros(out_f, cnt, dtype=torch.float32, device=W.device)
            Qi1 = torch.zeros(out_f, cnt, dtype=quant.code_dtype, device=W.device)
            Err1 = torch.zeros(out_f, cnt, dtype=torch.float32, device=W.device)
            H1 = Hp[i1:i2, i1:i2]
            for i in range(cnt):
                j = i1 + i
                w = W1[:, i]
                q_int, dq = quant.quantize_column(w, sc_cols[:, j], zr_cols[:, j])
                Q1[:, i] = dq
                Qi1[:, i] = q_int
                d = H1[i, i]
                err = (w - dq) / d
                W1[:, i:] -= torch.outer(err, H1[i, i:])
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            Qint[:, i1:i2] = Qi1
            if i2 < in_f:
                Tmat[:, i2:] -= Err1 @ Hp[i1:i2, i2:]

    # ---- 8. undo the permutation
    if cfg.act_order:
        Q = Q.index_select(1, invperm)
        Qint = Qint.index_select(1, invperm)

    W_hat = Q.to(W.dtype)
    obj = reconstruction_objective(Q, W_solve, H, Cw if use_asym else None)

    meta = {
        "method": "gptaq" if use_asym else "gptq",
        "bits": cfg.bits, "group_size": cfg.group_size, "sym": cfg.sym,
        "act_order": cfg.act_order, "true_sequential": cfg.true_sequential,
        "asym": use_asym, "damp": damp, "jitter_used": jitter_used,
        "n_dead_columns": n_dead, "n_tokens": H.shape[0],
        "objective": obj,
        "objective_per_token": obj / max(in_f, 1),
        "rel_fro_err": float((Q - W_solve).norm() / W_solve.norm().clamp_min(1e-12)),
        "max_abs_err": float((Q - W_solve).abs().max()),
        "hessian": diag,
        "time_s": round(time.time() - t0, 3),
    }
    if verbose:
        cond = diag.get("cond") or diag.get("cond_damped_est")
        cond_s = f"{cond:.2e}" if isinstance(cond, (int, float)) else "n/a"
        print(f"    {name:52s} {out_f:5d}x{in_f:<5d} "
              f"cond={cond_s} obj={obj:.3e} {meta['time_s']}s")
    return QuantizedLinear(name, Qint, scales, zeros, g_idx, W_hat, meta)


# ======================================================================================
# Activation capture
# ======================================================================================


def capture_inputs(wrapped, batch, modules: Dict[str, nn.Linear],
                   to_cpu_dtype=torch.float16) -> Dict[str, torch.Tensor]:
    """
    One forward pass, returning the input activation of each module as [in, T] on CPU.

    Hooks are removed immediately; nothing is retained on the GPU. The [1, T, in]
    activation is transposed to [in, T] so column t is exactly token t (B = 1).
    """
    buf: Dict[str, torch.Tensor] = {}
    handles = []

    def make(name):
        def hook(_mod, inp, _out):
            x = inp[0]
            if x.dim() == 3:
                x = x[0]
            buf[name] = x.detach().to(to_cpu_dtype).T.contiguous().cpu()
        return hook

    for name, mod in modules.items():
        handles.append(mod.register_forward_hook(make(name)))
    try:
        with torch.no_grad():
            wrapped.forward_hidden(batch.input_ids, batch.attention_mask)
    finally:
        for h in handles:
            h.remove()
    return buf


def capture_fp16_reference(wrapped, batch, modules, ref_cpu: Dict[str, torch.Tensor]
                           ) -> Dict[str, torch.Tensor]:
    """Forward pass with the original FP16 weights swapped in, capturing inputs."""
    swap_to_ref(wrapped, ref_cpu, len(wrapped.decoder_layers()))
    try:
        return capture_inputs(wrapped, batch, modules)
    finally:
        swap_to_current(wrapped)


# ======================================================================================
# Weight swapping (reference vs. partially quantized state)
# ======================================================================================


def swap_to_ref(wrapped, ref_cpu: Dict[str, torch.Tensor], upto: int) -> None:
    """Restore the original FP16 weights for layers 0..upto-1."""
    for layer_idx in range(min(upto, len(wrapped.decoder_layers()))):
        for name, mod in wrapped.layer_linears(layer_idx).items():
            if name in ref_cpu:
                mod.weight.data.copy_(ref_cpu[name].to(mod.weight.device))


def swap_to_current(wrapped, cur_cpu: Optional[Dict[str, torch.Tensor]] = None) -> None:
    """Copy back the quantized weights we have already produced."""
    if not cur_cpu:
        return
    for name, tens in cur_cpu.items():
        mod = wrapped.model.get_submodule(name)
        mod.weight.data.copy_(tens.to(mod.weight.device))


# ======================================================================================
# Model-level driver
# ======================================================================================


def quantize_model(
    wrapped,
    batches: Sequence,
    cfg: QuantConfig,
    verbose: bool = True,
    progress: Optional[Callable[[int, int, str], None]] = None,
    max_layers: Optional[int] = None,
) -> Dict[str, QuantizedLinear]:
    """
    Quantise every linear module of the backbone, layer by layer, in place.

    Two forward passes per layer when `cfg.drift_correction` is True:
      (a) with the original FP16 weights  -> X~  (the reference input)
      (b) with the layers < l already quantised -> X (the drifted input)
    That is the honest GPTAQ setting. Setting drift_correction=False collapses
    X~ := X, which is plain weighted GPTQ (the `asym=0`-style ablation of drift).

    Memory: reference weights and produced weights live in CPU RAM, so the GPU only
    ever holds the live model plus one layer's Hessian/cross-covariance.
    """
    n_layers = len(wrapped.decoder_layers())
    n_layers = min(n_layers, max_layers) if max_layers else n_layers

    ref_cpu: Dict[str, torch.Tensor] = {}
    if cfg.drift_correction:
        for l in range(n_layers):
            for name, mod in wrapped.layer_linears(l).items():
                ref_cpu[name] = mod.weight.detach().to(torch.float16).cpu()
        if verbose:
            total = sum(t.numel() for t in ref_cpu.values()) * 2 / 2 ** 30
            print(f"  cached FP16 reference weights: {total:.2f} GB in CPU RAM")

    results: Dict[str, QuantizedLinear] = {}
    cur_cpu: Dict[str, torch.Tensor] = {}
    t_start = time.time()

    for l in range(n_layers):
        linears = wrapped.layer_linears(l)
        names = [n for n, m in linears.items() if m.in_features >= cfg.min_in_features]
        mods = {n: linears[n] for n in names}
        if not names:
            continue

        accs = {
            n: WeightedHessian(mods[n].in_features, cfg.hessian_dtype,
                               track_cross=cfg.asym)
            for n in names
        }

        # (a) reference activations, if drift correction is on and the layer is not l=0
        need_ref = cfg.drift_correction and l > 0
        if need_ref:
            ref_buf: List[Dict[str, torch.Tensor]] = []
            swap_to_ref(wrapped, ref_cpu, l + 1)
            try:
                for b in batches:
                    ref_buf.append(capture_inputs(wrapped, b, mods))
            finally:
                swap_to_current(wrapped, cur_cpu)

        # (b) drifted activations under the current (partially quantized) model
        for bi, b in enumerate(batches):
            x = capture_inputs(wrapped, b, mods)
            w = b.weights
            for n in names:
                xr = ref_buf[bi][n] if need_ref else None
                accs[n].add(x[n], xr, w)

        if verbose:
            print(f"  layer {l:2d}/{n_layers-1}  tokens={accs[names[0]].n_tokens}  "
                  f"ess={accs[names[0]].ess_fraction():.3f}")

        # (c) solve each module
        diag_this_layer = (l % max(1, int(cfg.diag_every_n))) == 0
        for n in names:
            res = quantize_linear(n, mods[n].weight.detach(), accs[n].H, accs[n].Cw,
                                  cfg, verbose=verbose,
                                  compute_diagnostics=diag_this_layer)
            if res is None:
                continue
            wrapped.set_weight(n, res.W_hat)
            cur_cpu[n] = res.W_hat.detach().to(torch.float16).cpu()
            results[n] = res

        del accs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if progress is not None:
            progress(l, n_layers, f"layer {l} done")

    if verbose:
        print(f"  quantized {len(results)} modules in {time.time()-t_start:.1f}s")
    return results


def quantize_model_rtn(wrapped, cfg: QuantConfig, verbose: bool = True,
                       max_layers: Optional[int] = None) -> Dict[str, QuantizedLinear]:
    """Round-to-nearest baseline: same grid, no second-order solve."""
    n_layers = min(len(wrapped.decoder_layers()), max_layers or 10 ** 9)
    results: Dict[str, QuantizedLinear] = {}
    for l in range(n_layers):
        for name, mod in wrapped.layer_linears(l).items():
            res = quantize_linear(name, mod.weight.detach(), None, None, cfg, verbose=False)
            if res is None:
                continue
            wrapped.set_weight(name, res.W_hat)
            results[name] = res
    if verbose:
        print(f"  RTN applied to {len(results)} modules")
    return results


# ======================================================================================
# Checkpoint helpers
# ======================================================================================


def save_quant_state(results: Dict[str, QuantizedLinear], path: str, meta: Dict = None) -> str:
    payload = {name: r.state_dict() for name, r in results.items()}
    payload["__meta__"] = meta or {}
    torch.save(payload, path)
    return path


def load_quant_state(wrapped, path: str, apply: bool = True) -> Dict[str, QuantizedLinear]:
    """Re-apply a saved fake-quant checkpoint to a freshly loaded FP16 model."""
    payload = torch.load(path, map_location="cpu", weights_only=False)
    payload.pop("__meta__", None)
    results: Dict[str, QuantizedLinear] = {}
    for name, state in payload.items():
        res = QuantizedLinear(
            name=name,
            q_int=state["q_int"],
            scales=state["scales"],
            zeros=state["zeros"],
            g_idx=state["g_idx"],
            W_hat=state["W_hat"],
            meta=state.get("meta", {}),
        )
        results[name] = res
        if apply:
            wrapped.set_weight(name, res.W_hat)
    return results


def summarize_results(results: Dict[str, QuantizedLinear]) -> Dict[str, object]:
    by_module: Dict[str, List[float]] = {}
    for name, r in results.items():
        key = name.split(".")[-1]
        by_module.setdefault(key, []).append(float(r.meta.get("rel_fro_err", 0.0)))
    total_obj = sum(float(r.meta.get("objective", 0.0)) for r in results.values())
    conds = sorted(c for c in (r.meta.get("hessian", {}).get("cond")
                               for r in results.values()) if c is not None)
    median_cond = conds[len(conds) // 2] if conds else None
    return {
        "n_modules": len(results),
        "rel_fro_err_by_module": {k: round(sum(v) / len(v), 6) for k, v in by_module.items()},
        "total_objective": total_obj,
        "median_cond": median_cond,
        "max_cond": max(conds) if conds else None,
        "total_time_s": round(sum(float(r.meta.get("time_s", 0)) for r in results.values()), 1),
    }
'''
MODULES['quantizer_adapter.py'] = r'''"""
Integer-code export and packing for real low-bit kernels (thin adapter).

The solver deliberately never touches a kernel. It returns integer codes plus group
parameters, and this module is the only place that knows about GPTQ's on-disk
layout. That keeps the research code honest: everything measured through the
proxy gate is fake-quant, and turning a checkpoint into real INT3/INT4 inference is
a separate, verifiable step.

Two conventions are translated here, and both are easy to get wrong:

1. Zero representation.
   Our quantiser stores the *minimum* of each group (`zero` is a float offset):
        dequant = q * scale + zero
   GPTQ stores an integer zero-point z with
        dequant = (q - z) * scale
   so z = round(-zero / scale). If you forget this, asymmetric checkpoints come out
   with a constant shift and accuracy looks catastrophic for no reason.

2. Bit packing.
   `qweight` is int32, big-endian by bit order, with `32 // bits` consecutive input
   channels packed per word: word r of column c holds channels
   [r*words .. r*words + words - 1], low bits first.

`self_test()` unpacks the packed tensors and compares against the solver's own
dequantisation, so a layout mistake fails here instead of silently in a kernel.
"""
from __future__ import annotations

import json
import os
from typing import Dict, List, Optional

import torch

from .logging_utils import save_json


def words_per_int(bits: int) -> int:
    if 32 % bits != 0:
        raise ValueError(f"bits={bits} does not divide 32; kernel packing needs 2,4,8,16")
    return 32 // bits


PACKABLE_BITS = (2, 4, 8, 16)


def pack_codes_along_in(q_int: torch.Tensor, bits: int) -> torch.Tensor:
    """
    Pack integer codes [out, in] into GPTQ's qweight layout [in/words, out] int32.

    Consecutive input channels share a word; the first channel occupies the low bits.
    """
    w = words_per_int(bits)
    out_f, in_f = q_int.shape
    if in_f % w != 0:
        raise ValueError(f"in_features={in_f} is not a multiple of {w}")

    vals = q_int.to(torch.int64).transpose(0, 1).contiguous()   # [in, out]
    packed = torch.zeros(in_f // w, out_f, dtype=torch.int32)
    mask = (1 << bits) - 1
    for i in range(w):
        chunk = (vals[i::w] & mask).to(torch.int32)
        packed |= chunk << (bits * i)
    return packed


def unpack_codes_along_in(packed: torch.Tensor, bits: int, out_f: int) -> torch.Tensor:
    """Inverse of pack_codes_along_in. Returns [out, in] int64 codes."""
    w = words_per_int(bits)
    mask = (1 << bits) - 1
    rows = packed.shape[0]
    in_f = rows * w
    vals = torch.zeros(in_f, out_f, dtype=torch.int64)
    p = packed.to(torch.int64)
    for i in range(w):
        vals[i::w] = (p >> (bits * i)) & mask
    return vals.transpose(0, 1).contiguous()


def to_zero_point_form(zeros: torch.Tensor, scales: torch.Tensor, bits: int,
                       sym: bool) -> torch.Tensor:
    """Convert our float-min `zero` to GPTQ's integer zero-point."""
    if sym:
        return torch.full_like(zeros, (1 << (bits - 1)))
    qmax = float(2 ** bits - 1)
    z = torch.round(-zeros.to(torch.float32) / scales.to(torch.float32).clamp_min(1e-12))
    return z.clamp(0, qmax).to(torch.int32)


def export_module(result, bits: int, sym: bool) -> Dict[str, torch.Tensor]:
    """
    One QuantizedLinear -> GPTQ-style tensors plus our own dequant reference.

    For bit widths that do not divide 32 (3, 5, 6, 7) the integer codes are exported
    unpacked as int8 under `q_int`, because GPTQ's strided int32 layout has no exact
    3-bit form: real 3-bit kernels use a different stride/padding convention that has
    to be validated against the specific loader. Fake-quant evaluation (which is what
    the proxy gate uses) needs nothing more than `q_int` + group parameters.
    """
    z_int = to_zero_point_form(result.zeros, result.scales, bits, sym)
    if sym:
        # GPTQ stores symmetric codes offset by 2^(b-1) with a zero-point of the same
        # value, so that dequant is (q_stored - z)*scale for both grid types.
        offset = 1 << (bits - 1)
        codes_store = (result.q_int.to(torch.int32) + offset)
        z_int = torch.full_like(result.zeros, float(offset)).to(torch.int32)
    else:
        codes_store = result.q_int.to(torch.int32)

    out: Dict[str, torch.Tensor] = {
        "scales": result.scales.to(torch.float16).contiguous(),      # [out, G]
        "g_idx": result.g_idx.to(torch.int32),
        "W_hat": result.W_hat.detach().to(torch.float16),            # fake-quant reference
        "q_int": codes_store.to(torch.int8).cpu(),                   # always available
    }
    if bits in PACKABLE_BITS:
        out["qweight"] = pack_codes_along_in(codes_store, bits)       # [in/w, out]
        out["qzeros"] = pack_codes_along_in(z_int.transpose(0, 1).contiguous(), bits)
        out["layout"] = torch.tensor([0], dtype=torch.int8)           # 0 = packed
    else:
        out["qzeros_int"] = z_int.cpu()
        out["layout"] = torch.tensor([1], dtype=torch.int8)           # 1 = unpacked int8
    return out


def self_test(result, bits: int, sym: bool) -> Dict[str, float]:
    """
    Round-trip: pack -> unpack -> dequantise, and compare with the solver's W_hat.

    A large error here means a layout/sign bug, not a modelling problem.
    """
    exp = export_module(result, bits, sym)
    out_f, in_f = result.q_int.shape

    if "qweight" in exp:
        codes = unpack_codes_along_in(exp["qweight"], bits, out_f)
        code_match = float((codes == exp["q_int"].to(torch.int64)).float().mean())
    else:
        codes = exp["q_int"].to(torch.int64)
        code_match = 1.0

    # rebuild dequantised weights from the *exported* representation
    g = result.g_idx.long()
    sc = result.scales.index_select(1, g)
    z_src = exp.get("qzeros_int", None)
    if z_src is None:
        z_int = to_zero_point_form(result.zeros, result.scales, bits, sym)
    else:
        z_int = z_src
    z = z_int.index_select(1, g)
    W_rt = (codes.to(torch.float32) - z.to(torch.float32)) * sc
    denom = result.W_hat.abs().mean().clamp_min(1e-8)
    return {
        "code_roundtrip_exact_frac": code_match,
        "layout": "packed" if "qweight" in exp else "unpacked-int8",
        "dequant_max_abs_err": float((W_rt - result.W_hat.to(torch.float32)).abs().max()),
        "dequant_rel_err": float(((W_rt - result.W_hat.to(torch.float32)).abs().mean()
                                 / denom)),
    }


def save_packed_checkpoint(results: Dict, out_dir: str, bits: int, sym: bool,
                           meta: Optional[Dict] = None) -> str:
    """Write one safetensors/torch file per module plus a config.json describing it."""
    os.makedirs(out_dir, exist_ok=True)
    manifest = {
        "format": "ebswgptaq-fakequant-v1",
        "bits": bits,
        "sym": sym,
        "note": ("Fake-quant checkpoint. Weights are stored as integer codes plus group "
                 "parameters; W_hat is the dequantised reference. Load with "
                 "ebswgptaq.quant_core.load_quant_state for proxy evaluation, or run "
                 "ebswgptaq.quantizer_adapter.export_for_kernel for real kernels."),
        "modules": {},
    }
    tensors: Dict[str, torch.Tensor] = {}
    for name, res in results.items():
        exp = export_module(res, bits, sym)
        safe = name.replace(".", "__")
        for key, value in exp.items():
            tensors[f"{safe}.{key}"] = value.cpu()
        manifest["modules"][name] = {"prefix": safe, "meta": res.meta}

    path = os.path.join(out_dir, "model.safetensors")
    try:
        from safetensors.torch import save_file

        save_file(tensors, path)
    except Exception:
        path = os.path.join(out_dir, "model.pt")
        torch.save(tensors, path)
    if meta:
        manifest["meta"] = meta
    save_json(manifest, os.path.join(out_dir, "quant_config.json"))
    return out_dir


def export_for_kernel(results: Dict, out_dir: str, base_model_id: str, bits: int,
                      sym: bool, desc_act: bool = False) -> str:
    """
    Placeholder for the real-kernel step (later phase).

    Deliberately raises: producing a loadable Marlin/GPTQModel checkpoint requires
    verified qzeros/qweight conventions for the exact library version in use, and
    an unverified export would produce a checkpoint that silently mis-dequantises.
    The saved fake-quant tensors are already in the shape the converter needs.

    When this phase is implemented:
      1. save_packed_checkpoint(...) here
      2. copy the base model's config.json / tokenizer files
      3. convert with GPTQModel: `GPTQModel.load(base_model_id, model_save_path)`
         or `python -m gptqmodel.export --model <dir> --format marlin`
      4. verify: dequantise the converted checkpoint and compare against `W_hat`
         with max_abs_err < 1e-3 (fp16 tolerance) before reporting any speed number
      5. only then report VRAM and tokens/sec (v2 §9.5: never mix compression
         metrics with throughput claims)
    """
    raise NotImplementedError(
        "Real packed kernels are a later phase (see LATER_PHASES.md, notebook 12). "
        "Use save_packed_checkpoint() to export the integer codes now. "
        f"Base model for the final export: {base_model_id}"
    )
'''
MODULES['score_weights.py'] = r'''"""
Turn per-position sensitivity scores into calibration weights.

Design rules taken from the pivot document
------------------------------------------
1. Weighting is a weighted least-squares objective, so weights are normalised to
   **mean 1** over the pool. That keeps the ridge damping on a comparable scale
   across variants (v2 §9.3).
2. The primary mapping is `w_t = exp(beta * z_t)` with a *modest fixed beta*, not a
   tuned one. A target-ESS bisection is available but is a secondary control, not
   the headline claim (v2 §4.2).
3. Controls must preserve the *distribution* of weights and break only the
   *alignment*: `shuffle_within_sequence` permutes weights inside each sequence,
   `random` draws weights from the same empirical distribution.
4. Sink positions (BOS / attention sink) are neutralised before standardisation,
   because their activation norm dominates the Gram matrix (v1 §2.3).
"""
from __future__ import annotations

import math
from typing import Dict, List, Optional, Sequence, Tuple

import torch

from .config import WeightConfig


def kish_ess(w: torch.Tensor) -> float:
    """Kish effective sample size (sum w)^2 / sum w^2."""
    wf = w.detach().to(torch.float32).flatten().clamp_min(0.0)
    s1 = wf.sum()
    s2 = wf.pow(2).sum()
    if s2 <= 0:
        return 0.0
    return float(s1 * s1 / s2)


def ess_fraction(w: torch.Tensor) -> float:
    n = w.numel()
    return kish_ess(w) / max(n, 1)


def concentration_stats(w: torch.Tensor) -> Dict[str, float]:
    """Everything the results table needs about the weight distribution."""
    wf = w.detach().to(torch.float32).flatten().clamp_min(0.0)
    n = wf.numel()
    if n == 0:
        return {}
    total = float(wf.sum())
    order = torch.sort(wf, descending=True).values
    top1 = max(1, int(round(0.01 * n)))
    top10 = max(1, int(round(0.10 * n)))
    return {
        "n": n,
        "mean": float(wf.mean()),
        "max": float(wf.max()),
        "median": float(wf.median()),
        "min": float(wf.min()),
        "p99": float(torch.quantile(wf, torch.tensor(0.99, dtype=torch.float32))),
        "ess": kish_ess(wf),
        "ess_fraction": ess_fraction(wf),
        "mass_top1pct": float(order[:top1].sum() / max(total, 1e-12)),
        "mass_top10pct": float(order[:top10].sum() / max(total, 1e-12)),
        "max_over_median": float(wf.max() / max(float(wf.median()), 1e-12)),
    }


def winsorize(z: torch.Tensor, quantile: float) -> torch.Tensor:
    if quantile is None or quantile <= 0.0 or quantile >= 1.0:
        return z
    hi = torch.quantile(z, torch.tensor(quantile, dtype=torch.float32))
    return torch.clamp(z, max=float(hi))


def standardize(z: torch.Tensor) -> torch.Tensor:
    mu = z.mean()
    sd = z.std(unbiased=False)
    if float(sd) < 1e-12:
        return torch.zeros_like(z)
    return (z - mu) / sd


def exp_weights(z: torch.Tensor, beta: float) -> torch.Tensor:
    """w = exp(beta*z) normalised to mean 1. Always positive, never zero."""
    w = torch.exp(float(beta) * z.double() if z.dtype == torch.float64 else float(beta) * z)
    m = w.mean()
    if float(m) <= 0:
        return torch.ones_like(z)
    return w / m


def solve_beta_for_ess(
    z: torch.Tensor,
    target_ess_fraction: float,
    beta_max: float = 50.0,
    iters: int = 30,
) -> Tuple[float, float]:
    """
    30-step bisection for the inverse temperature whose induced weights hit a target
    normalised ESS (v1 Theorem 1, retained here only as a secondary control).

    ESS_fraction(beta) is strictly decreasing for non-constant z, so bisection is
    valid. Returns (beta, achieved_ess_fraction).
    """
    if target_ess_fraction >= 1.0:
        return 0.0, 1.0
    if float(z.std(unbiased=False)) < 1e-12:
        return 0.0, 1.0

    zf = z.detach().to(torch.float64)
    n = zf.numel()
    m = int((zf == zf.max()).sum())
    floor = m / n
    if target_ess_fraction <= floor:
        return float(beta_max), floor

    lo, hi = 0.0, float(beta_max)

    def ess_at(beta: float) -> float:
        return ess_fraction(exp_weights(zf, beta))

    if ess_at(hi) > target_ess_fraction:
        return hi, ess_at(hi)

    beta = 0.5 * hi
    for _ in range(iters):
        beta = 0.5 * (lo + hi)
        if ess_at(beta) > target_ess_fraction:
            lo = beta
        else:
            hi = beta
    return beta, ess_at(beta)


def _apply_sink_mask(positions: torch.Tensor, valid: Optional[torch.Tensor],
                     cfg: WeightConfig) -> torch.Tensor:
    """Positions to neutralise: BOS/sink by default."""
    mask = torch.zeros_like(positions, dtype=torch.bool)
    if cfg.sink_mask:
        for p in cfg.sink_positions:
            mask |= positions == int(p)
    return mask


def build_weights(
    scores: torch.Tensor,
    positions: torch.Tensor,
    valid: Optional[torch.Tensor] = None,
    cfg: Optional[WeightConfig] = None,
    sequence_id: Optional[torch.Tensor] = None,
) -> Dict[str, object]:
    """
    Scores -> mean-1 weights for one sequence (or one whole pool if sequence_id=None).

    `scores` may contain NaN/negative values for positions with no valid target;
    those are given the neutral weight 1.0.
    """
    cfg = cfg or WeightConfig()
    s = scores.detach().to(torch.float32).flatten().clone()
    pos = positions.flatten()

    finite = torch.isfinite(s)
    if valid is not None:
        finite &= valid.flatten().bool()
    if cfg.signal == "none":
        return {
            "weights": torch.ones_like(s),
            "beta": 0.0,
            "stats": concentration_stats(torch.ones_like(s)),
            "note": "uniform baseline",
        }

    # neutralise sink positions by masking them out of the statistics
    sink = _apply_sink_mask(pos, valid, cfg)
    stat_mask = finite & (~sink)

    if int(stat_mask.sum()) < 4:
        return {
            "weights": torch.ones_like(s),
            "beta": 0.0,
            "stats": concentration_stats(torch.ones_like(s)),
            "note": "too few valid scores; fell back to uniform",
        }

    ref = s[stat_mask]
    s = torch.where(finite, s, ref.median())
    s = winsorize(s, cfg.clip_quantile)
    z = standardize(s) if cfg.standardize else s

    if cfg.beta is not None:
        beta, achieved = float(cfg.beta), ess_fraction(exp_weights(z, cfg.beta))
    elif cfg.ess_floor is not None and cfg.ess_floor < 1.0:
        beta, achieved = solve_beta_for_ess(z, float(cfg.ess_floor), cfg.beta_max)
    else:
        beta, achieved = 0.0, 1.0

    w = exp_weights(z, beta)

    # neutral positions get exactly the mean weight (1.0)
    neutral = (~finite) | sink
    if neutral.any():
        w = torch.where(neutral, torch.ones_like(w), w)
        w = w / w.mean()

    if cfg.shuffle_within_sequence:
        g = torch.Generator().manual_seed(int(cfg.seed))
        idx = torch.randperm(w.numel(), generator=g)
        w = w[idx]

    return {
        "weights": w,
        "beta": beta,
        "achieved_ess_fraction": achieved,
        "stats": concentration_stats(w),
        "z_stats": {
            "mean": float(z.mean()),
            "std": float(z.std(unbiased=False)),
            "min": float(z.min()),
            "max": float(z.max()),
        },
    }


def random_weights_like(w: torch.Tensor, seed: int = 0) -> torch.Tensor:
    """
    Negative control: same *distribution* of weights, no alignment to the signal.

    Draws a permutation of the empirical weights, which preserves the exact marginal
    distribution (and therefore the concentration stats) while destroying any
    correlation with position (v2 §5.2).
    """
    g = torch.Generator().manual_seed(int(seed))
    flat = w.detach().flatten().clone()
    return flat[torch.randperm(flat.numel(), generator=g)].reshape(w.shape)


def per_sequence_weights(
    scores_per_seq: Sequence[torch.Tensor],
    positions_per_seq: Sequence[torch.Tensor],
    cfg: WeightConfig,
    valid_per_seq: Optional[Sequence[torch.Tensor]] = None,
) -> List[torch.Tensor]:
    """Independent normalisation inside each sequence (the `normalize=per_sequence` mode)."""
    out = []
    for i, (s, p) in enumerate(zip(scores_per_seq, positions_per_seq)):
        v = valid_per_seq[i] if valid_per_seq is not None else None
        out.append(build_weights(s, p, v, cfg)["weights"])
    return out
'''
MODULES['statistics.py'] = r'''"""
Statistics for the proxy gate: paired uncertainty and rank agreement.

All comparisons in this project are *paired*: the same held-out sequences, the same
tokens, the same decoding settings. So the unit of resampling is the sequence (or
problem), never the individual token -- token-level bootstrapping would produce
absurdly narrow intervals because tokens are strongly dependent within a sequence.
"""
from __future__ import annotations

import math
from typing import Dict, List, Optional, Sequence, Tuple

import torch


def paired_bootstrap_ci(
    deltas: Sequence[float],
    n_resamples: int = 10000,
    alpha: float = 0.05,
    seed: int = 0,
) -> Dict[str, float]:
    """
    Percentile bootstrap CI for the mean of paired differences.

    `deltas[i]` = metric(method) - metric(baseline) for sequence/problem i.
    A CI that excludes 0 is the evidence bar for the specificity gate.
    """
    d = torch.as_tensor(list(deltas), dtype=torch.float32).flatten()
    n = d.numel()
    if n == 0:
        return {}
    g = torch.Generator().manual_seed(int(seed))
    idx = torch.randint(0, n, (n_resamples, n), generator=g)
    means = d[idx].mean(dim=1)
    lo = torch.quantile(means, alpha / 2)
    hi = torch.quantile(means, 1 - alpha / 2)
    return {
        "n": n,
        "mean": float(d.mean()),
        "std": float(d.std(unbiased=True)) if n > 1 else 0.0,
        "ci_lo": float(lo),
        "ci_hi": float(hi),
        "alpha": alpha,
        "n_resamples": n_resamples,
        "excludes_zero": bool(float(lo) > 0 or float(hi) < 0),
        "p_direction": float((means > 0).float().mean()),
    }


def mcnemar_exact(baseline_correct: Sequence[bool], method_correct: Sequence[bool]
                  ) -> Dict[str, float]:
    """
    Exact McNemar test on paired per-item correctness.

    b = baseline right & method wrong, c = baseline wrong & method right.
    Two-sided exact binomial p-value on min(b, c) with n = b + c.
    """
    b = sum(1 for x, y in zip(baseline_correct, method_correct) if x and not y)
    c = sum(1 for x, y in zip(baseline_correct, method_correct) if (not x) and y)
    n = b + c
    if n == 0:
        return {"b": 0, "c": 0, "n": 0, "p_value": 1.0, "statistic": 0.0}
    k = min(b, c)
    # exact two-sided binomial with p=0.5
    tail = sum(math.comb(n, i) for i in range(0, k + 1)) / (2 ** n)
    p = min(1.0, 2.0 * tail)
    return {
        "b": b, "c": c, "n": n,
        "p_value": float(p),
        "statistic": float(abs(b - c)),
        "delta_accuracy": (c - b) / max(len(list(method_correct)), 1),
    }


def rank_correlation(x: Sequence[float], y: Sequence[float]) -> Dict[str, float]:
    """Spearman rho and Kendall tau between two per-position score vectors."""
    try:
        from scipy.stats import kendalltau, spearmanr
    except Exception as exc:  # pragma: no cover
        return {"error": f"scipy unavailable: {exc}"}

    a = torch.as_tensor(list(x), dtype=torch.float64).flatten().numpy()
    b = torch.as_tensor(list(y), dtype=torch.float64).flatten().numpy()
    if a.size < 3 or b.size < 3:
        return {"spearman": None, "kendall": None, "n": int(a.size)}
    rho, p_rho = spearmanr(a, b)
    tau, p_tau = kendalltau(a, b)
    return {
        "n": int(a.size),
        "spearman": float(rho),
        "spearman_p": float(p_rho),
        "kendall": float(tau),
        "kendall_p": float(p_tau),
    }


def topk_overlap(x: Sequence[float], y: Sequence[float], k: int) -> Dict[str, float]:
    """Jaccard overlap of the top-k position sets under two scores."""
    a = torch.as_tensor(list(x), dtype=torch.float32).flatten()
    b = torch.as_tensor(list(y), dtype=torch.float32).flatten()
    k = min(k, a.numel(), b.numel())
    if k == 0:
        return {"k": 0, "overlap": 0.0}
    sa = set(torch.topk(a, k).indices.tolist())
    sb = set(torch.topk(b, k).indices.tolist())
    return {
        "k": k,
        "overlap": len(sa & sb) / k,
        "jaccard": len(sa & sb) / max(len(sa | sb), 1),
        "chance": k / max(a.numel(), 1),
    }


def summarise_deltas(per_seq_method: Dict[str, Sequence[float]],
                     baseline_key: str,
                     n_resamples: int = 10000,
                     seed: int = 0) -> Dict[str, Dict[str, float]]:
    """Paired bootstrap of every variant against one baseline, keyed by variant."""
    base = list(per_seq_method[baseline_key])
    out: Dict[str, Dict[str, float]] = {}
    for key, values in per_seq_method.items():
        if key == baseline_key:
            continue
        values = list(values)
        if len(values) != len(base):
            out[key] = {"error": f"length mismatch {len(values)} vs {len(base)}"}
            continue
        deltas = [v - b for v, b in zip(values, base)]
        out[key] = paired_bootstrap_ci(deltas, n_resamples=n_resamples, seed=seed)
    return out
'''
MODULES['trace_generation.py'] = r'''"""
Self-generated reasoning traces for the calibration corpus (Milestone 1).

Why self-generated
------------------
Generic web text is distributionally far from a long chain of thought, and
quantising on it leaves the high-entropy fork positions under-represented. Okoshi et
al. and Zheng et al. both use self-generated trajectories; the pivot adopts that
without claiming it as novel.

Determinism
-----------
The primary corpus uses greedy decoding so the corpus can be regenerated bit-exactly
from (prompt_ids, model revision, seed). Prompts, full traces, token ids and every
decoding parameter are stored per record, because an unreproducible calibration set
invalidates every downstream comparison.
"""
from __future__ import annotations

import time
from typing import Dict, List, Optional, Sequence

import torch

from .config import CalibConfig


def _gen_kwargs(cfg: CalibConfig, max_new_tokens: Optional[int], tokenizer) -> Dict:
    kwargs = {
        "max_new_tokens": int(max_new_tokens or cfg.gen_max_new_tokens),
        "do_sample": bool(cfg.gen_do_sample),
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }
    if cfg.gen_do_sample:
        kwargs.update({"temperature": float(cfg.gen_temperature), "top_p": float(cfg.gen_top_p)})
    return kwargs


@torch.no_grad()
def generate_traces(
    wrapped,
    prompts: Sequence[Dict],
    cfg: CalibConfig,
    template: str = "plain",
    max_new_tokens: Optional[int] = None,
    verbose: bool = True,
) -> List[Dict]:
    """
    Generate one trace per prompt and return JSONL-ready records.

    Each record carries the exact token_ids of prompt+trace, so the calibration
    batches never re-tokenise.
    """
    from .calibration_data import render_prompt

    tokenizer = wrapped.tokenizer
    records: List[Dict] = []
    t0 = time.time()

    for i, item in enumerate(prompts):
        question = item["prompt"]
        rendered = render_prompt(question, template)
        enc = tokenizer(rendered, return_tensors="pt").to(wrapped.device)
        prompt_len = int(enc["input_ids"].shape[1])

        out = wrapped.generate(enc["input_ids"], attention_mask=enc["attention_mask"],
                               **_gen_kwargs(cfg, max_new_tokens, tokenizer))
        full_ids = out[0].tolist()
        new_ids = full_ids[prompt_len:]
        text = tokenizer.decode(new_ids, skip_special_tokens=True)

        hit_limit = len(new_ids) >= int(max_new_tokens or cfg.gen_max_new_tokens)
        records.append({
            "prompt_id": item["prompt_id"],
            "source": item.get("source", "unknown"),
            "source_split": item.get("source_split", "unknown"),
            "prompt": question,
            "rendered_prompt": rendered,
            "template": template,
            "trace": text,
            "token_ids": full_ids,
            "prompt_len": prompt_len,
            "n_new_tokens": len(new_ids),
            "hit_max_new_tokens": bool(hit_limit),
            "do_sample": bool(cfg.gen_do_sample),
            "temperature": float(cfg.gen_temperature) if cfg.gen_do_sample else None,
            "top_p": float(cfg.gen_top_p) if cfg.gen_do_sample else None,
            "seed": int(cfg.seed),
            "model_id": wrapped.cfg.model_id,
            "model_revision": wrapped.cfg.revision,
            "tokenizer_name": getattr(tokenizer, "name_or_path", None),
        })
        if verbose and (i + 1) % 4 == 0:
            print(f"  {i+1}/{len(prompts)} traces  "
                  f"({records[-1]['n_new_tokens']} new tokens, "
                  f"{time.time()-t0:.0f}s elapsed)")
    return records


@torch.no_grad()
def tokenize_anchors(wrapped, texts: Sequence[str], cfg: CalibConfig) -> List[Dict]:
    """General-language anchors: tokenised, truncated, no generation."""
    tokenizer = wrapped.tokenizer
    records: List[Dict] = []
    for i, text in enumerate(texts):
        ids = tokenizer(text, truncation=True, max_length=cfg.max_seq_len,
                        add_special_tokens=True)["input_ids"]
        if len(ids) < 8:
            continue
        records.append({
            "prompt_id": f"anchor:{i:03d}",
            "source": cfg.general_dataset,
            "source_split": cfg.general_split,
            "prompt": None,
            "rendered_prompt": None,
            "template": None,
            "trace": text,
            "token_ids": ids,
            "prompt_len": 0,
            "n_new_tokens": len(ids),
            "hit_max_new_tokens": False,
            "do_sample": False,
            "seed": int(cfg.seed),
            "model_id": wrapped.cfg.model_id,
            "model_revision": wrapped.cfg.revision,
        })
    return records


def filter_records(records: Sequence[Dict], min_tokens: int = 64,
                   max_tokens: Optional[int] = None) -> List[Dict]:
    """Drop degenerate (too short) traces; keep everything else for inspection."""
    out = []
    for r in records:
        n = len(r.get("token_ids") or [])
        if n < min_tokens:
            continue
        if max_tokens is not None and n > max_tokens:
            continue
        out.append(r)
    return out


def build_mixed_corpus(
    reasoning_records: Sequence[Dict],
    anchor_records: Sequence[Dict],
    cfg: CalibConfig,
    max_seq_len: Optional[int] = None,
) -> List[Dict]:
    """
    Assemble the mixed corpus and truncate every sequence to the calibration length.

    Truncation is from the LEFT for reasoning traces (keep the reasoning body, drop
    the tail of the problem statement only if it is enormous) and from the right for
    anchors. In practice most traces are shorter than the window, so this rarely
    matters -- but it is recorded per record either way.
    """
    max_len = int(max_seq_len or cfg.max_seq_len)
    corpus: List[Dict] = []

    for rec in list(reasoning_records) + list(anchor_records):
        ids = list(rec["token_ids"])
        original = len(ids)
        if original > max_len:
            if rec["prompt_len"] and rec["prompt_len"] < max_len // 2:
                # keep the prompt head and the most recent reasoning tokens
                head = ids[: rec["prompt_len"]]
                tail = ids[-(max_len - len(head)):]
                ids = head + tail
                mode = "head+tail"
            else:
                ids = ids[:max_len]
                mode = "right"
        else:
            mode = "none"
        out = dict(rec)
        out["token_ids"] = ids
        out["length_before_truncation"] = original
        out["truncation_mode"] = mode
        corpus.append(out)
    return corpus


def corpus_stats(corpus: Sequence[Dict]) -> Dict[str, object]:
    lens = [len(r["token_ids"]) for r in corpus]
    n_reason = sum(1 for r in corpus if r.get("prompt") is not None)
    return {
        "n_sequences": len(corpus),
        "n_reasoning": n_reason,
        "n_general": len(corpus) - n_reason,
        "reasoning_fraction": round(n_reason / max(len(corpus), 1), 3),
        "total_tokens": int(sum(lens)),
        "length_min": min(lens) if lens else 0,
        "length_max": max(lens) if lens else 0,
        "length_mean": round(sum(lens) / max(len(lens), 1), 1),
        "n_truncated": sum(1 for r in corpus if r.get("truncation_mode") not in (None, "none")),
        "n_hit_max_new_tokens": sum(1 for r in corpus if r.get("hit_max_new_tokens")),
    }
'''

CONFIGS = {}
CONFIGS['calibration.yaml'] = r'''# Calibration corpus (Milestone 1)
# Pilot scale from v2 §5.1: 16-32 sequences. The 128-sequence / 262k-token corpus of
# v1 §2.2 is the later target, not a prerequisite for the proxy gate.
#
# Sized so the token count clears ~2x the model's largest input dimension
# (d_in = 8960 for down_proj in the 1.5B model). Notebook 02 asserts this: below ~1x,
# the Hessian is structurally rank-deficient and the ridge dominates the solve.

n_reasoning: 20          # self-generated CoT traces from training-split prompts
n_general: 7             # C4 / WikiText anchors -> ~74/26 mix at these settings
n_holdout: 8             # disjoint prompts, never used for calibration
max_seq_len: 2048
seed: 0
min_trace_tokens: 64

# prompt sources (training splits only)
prompt_dataset: openai/gsm8k
prompt_config: main
prompt_split: train
prompt_field: question

general_dataset: wikitext
general_config: wikitext-2-raw-v1
general_split: train
general_field: text

# generation: deterministic greedy for the primary corpus
gen_max_new_tokens: 1280
gen_do_sample: false
gen_temperature: 0.6
gen_top_p: 0.95

ngram_leak_n: 13
'''
CONFIGS['experiments.yaml'] = r'''# The decisive experiment matrix (notebook 06-08).
#
# Every variant uses the identical calibration corpus, quantizer hyper-parameters and
# bit-width. The ONLY thing that changes is the weight vector -- which is the point
# of the specificity control (hypothesis H4).

baseline: uniform

variants:
  uniform:
    signal: none
    note: "uniform GPTAQ: w_t = 1. Primary baseline."

  jsd:
    signal: jsd
    probe: rtn
    clip_quantile: 0.99
    standardize: true
    normalize: global
    beta: 1.0
    ess_floor: null
    sink_mask: true
    note: "proposed signal, modest fixed beta=1.0 (v2 §4.2)."

  jsd_ess50:
    signal: jsd
    probe: rtn
    clip_quantile: 0.99
    standardize: true
    normalize: global
    beta: null
    ess_floor: 0.50
    sink_mask: true
    note: "ESS-bounded variant, secondary control (v1 Theorem 1)."

  entropy:
    signal: entropy
    probe: none
    clip_quantile: 0.99
    standardize: true
    normalize: global
    beta: 1.0
    ess_floor: null
    sink_mask: true
    note: "open-loop secondary signal: is a quantized probe even necessary?"

  kl:
    signal: kl
    probe: rtn
    clip_quantile: 0.99
    standardize: true
    normalize: global
    beta: 1.0
    ess_floor: null
    sink_mask: true
    note: "unbounded forward KL, heavy-tailed negative control (v2 §4.1)."

  random:
    signal: random
    probe: jsd
    note: "same weight distribution, no alignment. Controls for nonuniform weighting."

  shuffled:
    signal: jsd
    probe: rtn
    shuffle_within_sequence: true
    beta: 1.0
    note: "preserves within-sequence weight distribution, breaks position alignment."

probe_swap:
  probes: [rtn, gptq]
  metrics: [spearman, kendall, topk_overlap]

gates:
  signal:
    rule: "high-JSD vs low-JSD subspace overlap must differ from the matched random control in a majority of layers"
  specificity:
    rule: "jsd must beat uniform, random and shuffled on held-out teacher-forced KL, with paired 95% CI excluding zero"
  robustness:
    rule: "the ranking must survive swapping the RTN probe for a GPTQ probe"
'''
CONFIGS['model.yaml'] = r'''# EB-SW-GPTAQ — model under test
# Pinned so notebook 00 can record an exact reproduction manifest.
model_id: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
revision: null            # set to a commit sha once you have one you trust
dtype: float16            # T4 has no native BF16 tensor cores
fp32_lm_head: true        # guards against FP16 overflow in the 152k-way projection
device: cuda
attn_implementation: sdpa
trust_remote_code: false
max_seq_len: 2048
exclude_name_patterns: ["lm_head"]
'''
CONFIGS['probe_gptq.yaml'] = r'''# Preliminary probe: uniform-weight GPTQ on the same calibration set.
# drift_correction is off for cost; this is a documented deviation from full GPTAQ
# and it is what the probe-swap analysis (notebook 08) exists to characterise.

bits: 4
group_size: 128
sym: false
damp_percent: 0.01
act_order: false
true_sequential: false
blocksize: 128
working_dtype: float32

hessian_dtype: float32
min_tokens_per_layer: 256

asym: false
drift_correction: false

mse_clip: false
n_clip_grid: 20

exclude_name_patterns: ["lm_head"]
min_in_features: 16

diag_max_dim: 2560
diag_topk: 64
diag_every_n: 2
'''
CONFIGS['probe_rtn.yaml'] = r'''# Preliminary probe: round-to-nearest (no data, no Hessian).
# The probe only has to be cheap and fixed, not good (v2 §4.1).

bits: 4
group_size: 128
sym: false
damp_percent: 0.01
act_order: false
true_sequential: false
blocksize: 128
working_dtype: float32

hessian_dtype: float32
min_tokens_per_layer: 256

asym: false
drift_correction: false

mse_clip: false
n_clip_grid: 20

exclude_name_patterns: ["lm_head"]
min_in_features: 16

diag_max_dim: 2560
diag_topk: 64
diag_every_n: 2
'''
CONFIGS['quant_w3.yaml'] = r'''# W3A16 primary regime: the accuracy cliff this study is about.
# Identical to quant_w4.yaml except bits. Group size stayed at 128 so the only
# difference between the W4 and W3 runs is the bit-width under test.

bits: 3
group_size: 128
sym: false
damp_percent: 0.01
act_order: false
true_sequential: false
blocksize: 128
working_dtype: float32

hessian_dtype: float32
min_tokens_per_layer: 256

asym: true
drift_correction: true

mse_clip: false
n_clip_grid: 20

exclude_name_patterns: ["lm_head"]
min_in_features: 16

diag_max_dim: 2560
diag_topk: 64
diag_every_n: 2
'''
CONFIGS['quant_w4.yaml'] = r'''# W4A16 pipeline-validation regime (v2 §5.1: start at W4, go to W3 only after the
# pipeline is demonstrably correct and the baseline shows headroom).

bits: 4
group_size: 128
sym: false                # asymmetric grid with zero-point
damp_percent: 0.01
act_order: false
true_sequential: false
blocksize: 128
working_dtype: float32

hessian_dtype: float32
min_tokens_per_layer: 256

asym: true                # GPTAQ-style asymmetric reconstruction
drift_correction: true    # capture X~ from the FP16 model (2 passes per layer)

mse_clip: false
n_clip_grid: 20

exclude_name_patterns: ["lm_head"]
min_in_features: 16

diag_max_dim: 2560
diag_topk: 64
diag_every_n: 2
'''

for _name, _src in MODULES.items():
    with open(os.path.join(PKG_DIR, _name), "w", encoding="utf-8", newline="\n") as _fh:
        _fh.write(_src)
for _name, _src in CONFIGS.items():
    with open(os.path.join(CONFIG_DIR, _name), "w", encoding="utf-8", newline="\n") as _fh:
        _fh.write(_src)

if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
import ebswgptaq
print(f'library ready: v{ebswgptaq.__version__} from {ebswgptaq.__file__}')
print(f'{len(MODULES)} modules, {len(CONFIGS)} configs written to {_ROOT}')

In [ ]:
# Check the packages this project needs. Kaggle preinstalls all of them; this prints
# the exact versions that go into the reproduction manifest.
REQUIRED = ["torch", "transformers", "datasets", "accelerate", "safetensors", "yaml", "scipy"]
missing = []
print("package versions")
print("-" * 46)
for name in REQUIRED:
    try:
        mod = __import__(name)
        print(f"  {name:14s} {getattr(mod, '__version__', 'ok')}")
    except Exception:
        missing.append(name)
        print(f"  {name:14s} MISSING")

if missing:
    print("\nInstall the missing packages, then re-run this cell:")
    print("  !pip -q install " + " ".join(missing))
else:
    print("\nAll required packages are present.")

In [ ]:
# Hardware reality check. The whole study is budgeted for a single 16 GB T4, so the
# numbers below decide how big a calibration corpus and model you can afford.
import os, json, torch

print("torch", torch.__version__, "| cuda runtime", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        native_bf16 = p.major >= 8
        print(f"  GPU {i}: {p.name} | {p.total_memory/2**30:.1f} GB | "
              f"sm_{p.major}{p.minor} | native bf16 = {native_bf16}")
    free, total = torch.cuda.mem_get_info()
    print(f"  free now: {free/2**30:.2f} GB of {total/2**30:.2f} GB")
    if not (p.major >= 8):
        print("  -> No native BF16: run the model in FP16 with fp32_lm_head=True "
              "(the default in configs/model.yaml).")

try:
    pages = os.sysconf("SC_PHYS_PAGES")
    page_size = os.sysconf("SC_PAGE_SIZE")
    print(f"\nCPU RAM: {pages * page_size / 2**30:.1f} GB")
except Exception:
    pass
try:
    import shutil
    du = shutil.disk_usage("/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    print(f"Working disk: {du.free/2**30:.1f} GB free of {du.total/2**30:.1f} GB")
except Exception:
    pass

In [ ]:
# CPU-only correctness checks. These are the same assertions the repo's test suite
# runs; they cover the mathematics every later notebook depends on.
import torch
from ebswgptaq import jsd, quant_core, score_weights, quantizer_adapter
from ebswgptaq.config import QuantConfig, WeightConfig

torch.manual_seed(0)
checks = {}

print("1) JSD invariants (symmetry, bounds, self-distance, top-k convergence)")
s = jsd.assert_jsd_sanity(v=512)
for k, v in s.items():
    print(f"     {k:24s} {v}")
checks["jsd_symmetry"] = s["symmetry_max_abs_diff"] < 1e-6
checks["jsd_self_zero"] = s["self_jsd_max"] < 1e-9
checks["jsd_bounded"] = bool(s["within_bound"])

print("\n2) causal alignment (logits[t] scored against token_ids[t+1])")
ca = jsd.assert_causal_alignment()
print(f"     {ca}")
checks["causal_alignment"] = ca["all_pairs_correct"]

print("\n3) group quantizer round-trip")
W = torch.randn(128, 512) * 0.02
q = quant_core.GroupQuantizer(bits=4, group_size=128, sym=False)
sc, zr, gi = q.fit(W)
q_int, dq = q.quantize(W, sc, zr, gi)
rel = float((dq - W).norm() / W.norm())
print(f"     q_int range [{int(q_int.min())}, {int(q_int.max())}] | "
      f"groups={sc.shape[1]} | rel_fro_err={rel:.4f}")
checks["quantizer_roundtrip"] = 0.0 < rel < 0.2

print("\n4) token weights and Kish ESS")
w = score_weights.build_weights(torch.randn(1000) * 2, torch.arange(1000), None,
                                WeightConfig(signal="jsd", ess_floor=0.5))["weights"]
st = score_weights.concentration_stats(w)
print(f"     ess_frac={st['ess_fraction']:.3f} max={st['max']:.2f} "
      f"mass_top1%={st['mass_top1pct']:.3f}")
checks["ess_bounded"] = abs(st["ess_fraction"] - 0.5) < 0.03
checks["weights_mean_one"] = abs(st["mean"] - 1.0) < 1e-4

print("\n5) integer-code packing round-trip")
r = quant_core.quantize_linear("t", W, W.abs().T @ W.abs(), None,
                               QuantConfig(bits=4, group_size=64, asym=False))
rt = quantizer_adapter.self_test(r, 4, False)
print(f"     {rt}")
checks["packing_roundtrip"] = rt["code_roundtrip_exact_frac"] > 0.999

print("\n" + "-" * 60)
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print("-" * 60)
assert all(checks.values()), f"failed checks: {[k for k, v in checks.items() if not v]}"
print("Environment is fit to run the pipeline.")

In [ ]:
# Record the manifest. This file is the reproduction record for the whole study.
from ebswgptaq import logging_utils as lu

manifest = lu.environment_manifest({
    "notebook": "00_environment_check",
    "artifact_root": lu.write_root(),
    "config_dir": CONFIG_DIR,
    "library_version": __import__("ebswgptaq").__version__,
    "self_checks_passed": True,
})
lu.save_json(manifest, "env/env_report.json")
lu.log_event("00_environment_check", gpu=manifest.get("torch", {}).get("devices"))
print(json.dumps({k: manifest[k] for k in ("timestamp", "python", "platform")}, indent=2))
print("\nwrote", lu.out("env/env_report.json"))

## Handoff

Nothing downstream depends on this notebook, but re-run it after changing the
environment so the manifest always matches the run that produced the results.

Next: **01_generate_traces.ipynb**.